In [ ]:
import io, os, re, urllib3, requests, datetime, glob, json
import pandas as pd
import numpy as np
from datetime import date
from bs4 import BeautifulSoup
from dateutil.parser import parse
import xml.etree.ElementTree as ET
from io import BytesIO
from urllib.parse import urljoin

#warnings.simplefilter("ignore", category=pd.errors.SettingWithCopyWarning)
import warnings
warnings.filterwarnings('ignore')

# =========================================================
# Paths
# =========================================================
# Define local project directories (relative to current working directory)
d = f"{os.getcwd()}/"
PATH_RAW = os.path.join(d, "..", "raw")   # raw inputs (downloads / intermediate CSVs)
# Create folder if it does not exist
os.makedirs(PATH_RAW, exist_ok=True)
hdr = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537'}

# Central Bank

All statistics available here: https://www.central-bank.org.tt/statistics/data-centre/

## CPI

In [ ]:
url = "https://www.central-bank.org.tt/data-center-block/?title=consumer-price-index-index-values-monthly&id=36"

r = requests.get(url, headers=hdr, timeout=60)
r.raise_for_status()

soup = BeautifulSoup(r.text, "html.parser")

table = soup.select_one("table#table_1")
if table is None:
    raise ValueError("Could not find table_1 in the page HTML")

# headers
cols = [th.get_text(strip=True) for th in table.select("thead th")]

# rows (only real <tr> rows under tbody)
rows = []
for tr in table.select("tbody tr"):
    tds = [td.get_text(strip=True) for td in tr.select("td")]
    if tds:
        rows.append(tds)

df = pd.DataFrame(rows, columns=cols)

# clean types
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")

for c in df.columns[1:]:
    df[c] = pd.to_numeric(df[c].replace({"n.d.": np.nan, "n.a.": np.nan}), errors="coerce")

df = df.sort_values("Date").reset_index(drop=True)

def clean_col(c: str) -> str:
    c = c.lower()
    c = re.sub(r"\s*\([^)]*\)", "", c)                # remove ( ... )
    c = c.replace("consumer price index", "cpi")      # substitute phrase
    c = re.sub(r"&", "and", c)                        # optional: & -> and
    c = re.sub(r"[^a-z0-9\s_-]", "", c)               # remove special chars (keep _ and -)
    c = re.sub(r"[\s\-]+", "_", c)                    # spaces / hyphens -> _
    c = re.sub(r"_+", "_", c)                         # collapse multiple _
    c = c.strip("_")
    return c

df = df.rename(columns={c: clean_col(c) for c in df.columns})
df.dropna(axis=0, inplace=True, how='all')
df['date'] = pd.to_datetime(df['date'])

df.set_index("date", inplace=True)
df.sort_index(inplace=True)

print(f"Updated data range: {df.index.min()} to {df.index.max()}") 
df.to_csv(f"{PATH_RAW}/cbtt_cpi.csv")

## Fiscal

### Fiscal - Revenue (Monthly)

In [ ]:
df = pd.read_csv(f"{PATH_RAW}/cbtt_cg_rev.csv", parse_dates=True, index_col="date")
print(f"Previous data range: {df.index.min()} to {df.index.max()}") 

url = "https://www.central-bank.org.tt/data-center-block/?title=fiscal-revenue-monthly&id=90"

r = requests.get(url, headers=hdr, timeout=60)
r.raise_for_status()

soup = BeautifulSoup(r.text, "html.parser")

table = soup.select_one("table#table_1")
if table is None:
    raise ValueError("Could not find table_1 in the page HTML")

# headers
cols = [th.get_text(strip=True) for th in table.select("thead th")]

# rows (only real <tr> rows under tbody)
rows = []
for tr in table.select("tbody tr"):
    tds = [td.get_text(strip=True) for td in tr.select("td")]
    if tds:
        rows.append(tds)

df = pd.DataFrame(rows, columns=cols)

# clean types
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")

for c in df.columns[1:]:
    df[c] = pd.to_numeric(df[c].replace({"n.d.": np.nan, "n.a.": np.nan}), errors="coerce")

df = df.sort_values("Date").reset_index(drop=True)

def clean_col(c: str) -> str:
    c = c.lower()
    
    # remove parentheses
    c = re.sub(r"\s*\([^)]*\)", "", c)

    # core replacements
    replacements = {
        "central government": "cg",
        "total": "tot",
        "current": "cur",
        "capital": "cap",
        "revenue": "rev",
        "non tax": "ntax",
        "non-tax": "ntax",
        "tax revenue": "tax_rev",
        "income and profits": "inc_prof",
        "goods and services": "goods_serv",
        "international trade": "int_trade",
        "value added tax": "vat",
        "vat": "vat",
        "non energy": "neng",
        "non-energy": "neng",
        "non oil": "noil",
        "non-oil": "noil",
        "energy": "eng",
        "oil": "oil",
        "sector": ""
    }

    for k, v in replacements.items():
        c = c.replace(k, v)

    # clean symbols
    c = re.sub(r"&", "and", c)
    c = re.sub(r"[^a-z0-9\s_-]", "", c)
    c = re.sub(r"[\s\-]+", "_", c)
    c = re.sub(r"_+", "_", c)
    c = c.strip("_")

    return c

df = df.rename(columns={c: clean_col(c) for c in df.columns})

df['date'] = pd.to_datetime(df['date'])

df.set_index("date", inplace=True)
df.sort_index(inplace=True)

df.dropna(axis=0, inplace=True, how='all')
print(f"Updated data range: {df.index.min()} to {df.index.max()}") 
df.to_csv(f"{PATH_RAW}/cbtt_cg_rev.csv")

### Fiscal - Expenditure (Monthly)

In [ ]:
df = pd.read_csv(f"{PATH_RAW}/cbtt_cg_exp.csv", parse_dates=True, index_col="date")
print(f"Previous data range: {df.index.min()} to {df.index.max()}") 

url = "https://www.central-bank.org.tt/data-center-block?title=fiscal-expenditure-monthly&id=94"

r = requests.get(url, headers=hdr, timeout=60)
r.raise_for_status()

soup = BeautifulSoup(r.text, "html.parser")

table = soup.select_one("table#table_1")
if table is None:
    raise ValueError("Could not find table_1 in the page HTML")

# headers
cols = [th.get_text(strip=True) for th in table.select("thead th")]

# rows (only real <tr> rows under tbody)
rows = []
for tr in table.select("tbody tr"):
    tds = [td.get_text(strip=True) for td in tr.select("td")]
    if tds:
        rows.append(tds)

df = pd.DataFrame(rows, columns=cols)

# clean types
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")

for c in df.columns[1:]:
    df[c] = pd.to_numeric(df[c].replace({"n.d.": np.nan, "n.a.": np.nan}), errors="coerce")

df = df.sort_values("Date").reset_index(drop=True)

def clean_col(c: str) -> str:
    c = c.lower()
    c = re.sub(r"\s*\([^)]*\)", "", c)            # remove ( ... )
    
    # Abbreviations
    c = c.replace("central government", "cg")
    c = c.replace("expenditure on", "")
    c = c.replace("expenditure", "exp")
    c = c.replace("transfers", "trans")
    c = c.replace("subsidies", "subs")
    c = c.replace("and net lending", "cap_net_lending")
    c = c.replace("interest payments", "int_payments")
    c = c.replace("wages and salaries", "wages_salaries")
    c = c.replace("goods & services", "goods_services")
    c = c.replace("non-profit insitutions", "npi")
    c = c.replace("educational insitutions", "edu_inst")
    c = c.replace("state enterprises", "state_enterprises")
    c = c.replace("statutory boards and similar bodies", "stat_boards")
    c = c.replace("pensions and gratuities", "pensions_gratuities")
    c = c.replace("fuel subsidy payments", "fuel_subsidy")
    c = c.replace("senior citizens grants", "senior_citizens_grants")

    c = re.sub(r"&", "and", c)
    c = re.sub(r"[^a-z0-9\s_-]", "", c)
    c = re.sub(r"[\s\-]+", "_", c)
    c = re.sub(r"_+", "_", c)
    c = c.strip("_")
    
    return c

df = df.rename(columns={c: clean_col(c) for c in df.columns})

df['date'] = pd.to_datetime(df['date'])

df.set_index("date", inplace=True)
df.sort_index(inplace=True)

df.dropna(axis=0, inplace=True, how='all')
print(f"Updated data range: {df.index.min()} to {df.index.max()}") 
df.to_csv(f"{PATH_RAW}/cbtt_cg_exp.csv")

### Net financing

In [ ]:
df = pd.read_csv(f"{PATH_RAW}/cbtt_cg_fin.csv", parse_dates=True, index_col="date")
print(f"Previous data range: {df.index.min()} to {df.index.max()}") 

url = "https://www.central-bank.org.tt/data-center-block?title=fiscal-financing-monthly&id=102"

r = requests.get(url, headers=hdr, timeout=60)
r.raise_for_status()

soup = BeautifulSoup(r.text, "html.parser")

table = soup.select_one("table#table_1")
if table is None:
    raise ValueError("Could not find table_1 in the page HTML")

# headers
cols = [th.get_text(strip=True) for th in table.select("thead th")]

# rows (only real <tr> rows under tbody)
rows = []
for tr in table.select("tbody tr"):
    tds = [td.get_text(strip=True) for td in tr.select("td")]
    if tds:
        rows.append(tds)

df = pd.DataFrame(rows, columns=cols)

# clean types
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")

for c in df.columns[1:]:
    df[c] = pd.to_numeric(df[c].replace({"n.d.": np.nan, "n.a.": np.nan}), errors="coerce")

df = df.sort_values("Date").reset_index(drop=True)

def clean_col(c: str) -> str:
    c = c.lower()
    c = re.sub(r"\s*\([^)]*\)", "", c)  # remove (TT$Mn)
    
    # Abbreviations
    c = c.replace("total net financing", "cg_tot_net_fin")
    c = c.replace("net external financing", "cg_net_ext_fin")
    c = c.replace("external borrowing", "cg_ext_borrowing")
    c = c.replace("withdrawals from hsf", "cg_withdrawals_hsf")
    c = c.replace("external capital repayments", "cg_ext_cap_repayments")
    c = c.replace("net domestic financing", "cg_net_dom_fin")
    c = c.replace("domestic borrowing", "cg_dom_borrowing")
    c = c.replace("domestic capital repayments", "cg_dom_cap_repayments")
    
    c = re.sub(r"&", "and", c)
    c = re.sub(r"[^a-z0-9\s_-]", "", c)
    c = re.sub(r"[\s\-]+", "_", c)
    c = re.sub(r"_+", "_", c)
    c = c.strip("_")
    
    return c

df = df.rename(columns={c: clean_col(c) for c in df.columns})

df['date'] = pd.to_datetime(df['date'])

df.set_index("date", inplace=True)
df.sort_index(inplace=True)

df.dropna(axis=0, inplace=True, how='all')

print(f"Updated data range: {df.index.min()} to {df.index.max()}") 
df.to_csv(f"{PATH_RAW}/cbtt_cg_fin.csv")

### Balance and Ratios

In [ ]:
df = pd.read_csv(f"{PATH_RAW}/cbtt_cg_bal.csv", parse_dates=True, index_col="date")
print(f"Previous data range: {df.index.min()} to {df.index.max()}") 

url = "https://www.central-bank.org.tt/data-center-block/?title=fiscal-fiscal-balances-and-ratios-monthly&id=98"

r = requests.get(url, headers=hdr, timeout=60)
r.raise_for_status()

soup = BeautifulSoup(r.text, "html.parser")

table = soup.select_one("table#table_1")
if table is None:
    raise ValueError("Could not find table_1 in the page HTML")

# headers
cols = [th.get_text(strip=True) for th in table.select("thead th")]

# rows (only real <tr> rows under tbody)
rows = []
for tr in table.select("tbody tr"):
    tds = [td.get_text(strip=True) for td in tr.select("td")]
    if tds:
        rows.append(tds)

df = pd.DataFrame(rows, columns=cols)

# clean types
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")

for c in df.columns[1:]:
    df[c] = pd.to_numeric(df[c].replace({"n.d.": np.nan, "n.a.": np.nan}), errors="coerce")

df = df.sort_values("Date").reset_index(drop=True)

def clean_col(c: str) -> str:
    c = c.lower()
    c = re.sub(r"\s*\([^)]*\)", "", c)  # remove (TT$Mn) and (+)/(-)
    
    # Standardize fiscal terms
    c = c.replace("current account surplus / deficit", "cg_current_account_balance")
    c = c.replace("current account surplus+ / deficit -", "cg_current_account_balance")
    c = c.replace("non-energy fiscal balance", "cg_nonenergy_fiscal_balance")
    c = c.replace("primary balance", "cg_primary_balance")
    c = c.replace("overall fiscal balance", "cg_overall_fiscal_balance")
    
    c = re.sub(r"[^a-z0-9\s_-]", "", c)
    c = re.sub(r"[\s\-]+", "_", c)
    c = re.sub(r"_+", "_", c)
    c = c.strip("_")
    
    return c

df = df.rename(columns={c: clean_col(c) for c in df.columns})

df['date'] = pd.to_datetime(df['date'])

df.set_index("date", inplace=True)
df.sort_index(inplace=True)

df.dropna(axis=0, inplace=True, how='all')

print(f"Updated data range: {df.index.min()} to {df.index.max()}") 
df.to_csv(f"{PATH_RAW}/cbtt_cg_bal.csv")

## Motor Vehicle

### Registration

In [ ]:
df = pd.read_csv(f"{PATH_RAW}/cbtt_vehicles.csv", parse_dates=True, index_col="date")
print(f"Previous data range: {df.index.min()} to {df.index.max()}") 


url = "https://www.central-bank.org.tt/data-center-block/?title=output-motor-vehicle-registration-monthly&id=155"

r = requests.get(url, headers=hdr, timeout=60)
r.raise_for_status()

soup = BeautifulSoup(r.text, "html.parser")

table = soup.select_one("table#table_1")
if table is None:
    raise ValueError("Could not find table_1 in the page HTML")

# headers
cols = [th.get_text(strip=True) for th in table.select("thead th")]

# rows (only real <tr> rows under tbody)
rows = []
for tr in table.select("tbody tr"):
    tds = [td.get_text(strip=True) for td in tr.select("td")]
    if tds:
        rows.append(tds)

df = pd.DataFrame(rows, columns=cols)

# clean types
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")

for c in df.columns[1:]:
    df[c] = pd.to_numeric(df[c].replace({"n.d.": np.nan, "n.a.": np.nan}), errors="coerce")

df = df.sort_values("Date").reset_index(drop=True)

def clean_col(c: str) -> str:
    c = c.lower()
    
    # Remove redundant phrases
    c = c.replace("total number of ", "")
    c = c.replace(" vehicles registered", "_registered")
    c = c.replace(" vehicle registered", "_registered")
    
    # Standardize wording
    c = c.replace("private ", "private_")
    c = c.replace("commercial ", "commercial_")
    c = c.replace("new ", "new_")
    c = c.replace("used ", "used_")
    
    # Clean characters
    c = re.sub(r"[^a-z0-9\s_-]", "", c)
    c = re.sub(r"[\s\-]+", "_", c)
    c = re.sub(r"_+", "_", c)
    c = c.strip("_")
    
    return c

df = df.rename(columns={c: clean_col(c) for c in df.columns})


df['date'] = pd.to_datetime(df['date'])

df.set_index("date", inplace=True)
df.sort_index(inplace=True)

df.dropna(axis=0, inplace=True, how='all')

print(f"Updated data range: {df.index.min()} to {df.index.max()}") 

df.to_csv(f"{PATH_RAW}/cbtt_vehicles.csv")

### Sales

In [ ]:
df = pd.read_csv(f"{PATH_RAW}/cbtt_vehicles_sales.csv", parse_dates=True, index_col="date")
print(f"Previous data range: {df.index.min()} to {df.index.max()}") 


url = "https://www.central-bank.org.tt/data-center-block/?title=output-new-motor-vehicle-sales-monthly&id=153"


r = requests.get(url, headers=hdr, timeout=60)
r.raise_for_status()

soup = BeautifulSoup(r.text, "html.parser")

table = soup.select_one("table#table_1")
if table is None:
    raise ValueError("Could not find table_1 in the page HTML")

# headers
cols = [th.get_text(strip=True) for th in table.select("thead th")]

# rows (only real <tr> rows under tbody)
rows = []
for tr in table.select("tbody tr"):
    tds = [td.get_text(strip=True) for td in tr.select("td")]
    if tds:
        rows.append(tds)

df = pd.DataFrame(rows, columns=cols)

# clean types
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")

for c in df.columns[1:]:
    df[c] = pd.to_numeric(df[c].replace({"n.d.": np.nan, "n.a.": np.nan}), errors="coerce")

df = df.sort_values("Date").reset_index(drop=True)

def clean_col(c: str) -> str:
    c = c.lower()
    c = c.replace("number of ", "")
    
    # Turn "(commercial)" / "(private)" into suffixes
    c = re.sub(r"\(([^)]*)\)", r" \1", c)     # "(commercial)" -> " commercial"
    
    # Standardize key phrases
    c = c.replace("motor vehicles", "vehicles")
    c = c.replace("vehicle", "vehicles")
    c = c.replace("sold", "sales")
    c = c.replace("total", "tot")
    c = c.replace("new", "new")
    c = c.replace("commercial", "comm")
    c = c.replace("private", "priv")
    
    # Clean characters -> snake_case
    c = re.sub(r"[^a-z0-9\s_-]", "", c)
    c = re.sub(r"[\s\-]+", "_", c)
    c = re.sub(r"_+", "_", c)
    c = c.strip("_")
    
    # Add prefix (except date)
    if c == "date":
        return "date"
    return f"veh_{c}"

df = df.rename(columns={c: clean_col(c) for c in df.columns})


df['date'] = pd.to_datetime(df['date'])

df.set_index("date", inplace=True)
df.sort_index(inplace=True)

df.dropna(axis=0, inplace=True, how='all')

print(f"Updated data range: {df.index.min()} to {df.index.max()}") 

df.to_csv(f"{PATH_RAW}/cbtt_vehicles_sales.csv")

## Output

### Agricultural Output

In [ ]:
df = pd.read_csv(f"{PATH_RAW}/cbtt_agprod.csv", parse_dates=True, index_col="date")
print(f"Previous data range: {df.index.min()} to {df.index.max()}") 


url = "https://www.central-bank.org.tt/data-center-block/?title=output-production-of-agriculture-commodities-monthly&id=151"


r = requests.get(url, headers=hdr, timeout=60)
r.raise_for_status()

soup = BeautifulSoup(r.text, "html.parser")

table = soup.select_one("table#table_1")
if table is None:
    raise ValueError("Could not find table_1 in the page HTML")

# headers
cols = [th.get_text(strip=True) for th in table.select("thead th")]

# rows (only real <tr> rows under tbody)
rows = []
for tr in table.select("tbody tr"):
    tds = [td.get_text(strip=True) for td in tr.select("td")]
    if tds:
        rows.append(tds)

df = pd.DataFrame(rows, columns=cols)

# clean types
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")

for c in df.columns[1:]:
    df[c] = pd.to_numeric(df[c].replace({"n.d.": np.nan, "n.a.": np.nan}), errors="coerce")

df = df.sort_values("Date").reset_index(drop=True)

def clean_col(c: str) -> str:
    c = c.lower()
    
    # Remove units in parentheses
    c = re.sub(r"\s*\([^)]*\)", "", c)
    
    # Remove redundant phrases
    c = c.replace("production of ", "")
    
    # Standardize crop names
    c = c.replace("pigeon peas", "pigeon_peas")
    c = c.replace(" ", "_")
    
    # Clean special characters
    c = re.sub(r"[^a-z0-9_]", "", c)
    c = re.sub(r"_+", "_", c)
    c = c.strip("_")
    
    # Keep date unchanged
    if c == "date":
        return "date"
    
    return f"ag_prod_{c}"

df = df.rename(columns={c: clean_col(c) for c in df.columns})


df['date'] = pd.to_datetime(df['date'])

df.set_index("date", inplace=True)
df.sort_index(inplace=True)

df.dropna(axis=0, inplace=True, how='all')

print(f"Updated data range: {df.index.min()} to {df.index.max()}") 

df.to_csv(f"{PATH_RAW}/cbtt_agprod.csv")


### Output - Production, Exports and Local Sales of Cement (Monthly)

In [ ]:
df = pd.read_csv(f"{PATH_RAW}/cbtt_cement.csv", parse_dates=True, index_col="date")
print(f"Previous data range: {df.index.min()} to {df.index.max()}") 


url = "https://www.central-bank.org.tt/data-center-block/?title=output-production-exports-and-local-sales-of-cement-monthly&id=157"

r = requests.get(url, headers=hdr, timeout=60)
r.raise_for_status()

soup = BeautifulSoup(r.text, "html.parser")

table = soup.select_one("table#table_1")
if table is None:
    raise ValueError("Could not find table_1 in the page HTML")

# headers
cols = [th.get_text(strip=True) for th in table.select("thead th")]

# rows (only real <tr> rows under tbody)
rows = []
for tr in table.select("tbody tr"):
    tds = [td.get_text(strip=True) for td in tr.select("td")]
    if tds:
        rows.append(tds)

df = pd.DataFrame(rows, columns=cols)

# clean types
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")

for c in df.columns[1:]:
    df[c] = pd.to_numeric(df[c].replace({"n.d.": np.nan, "n.a.": np.nan}), errors="coerce")

df = df.sort_values("Date").reset_index(drop=True)

def clean_col(c: str) -> str:
    c = c.lower()
    
    # remove units in parentheses
    c = re.sub(r"\s*\([^)]*\)", "", c)
 
    # standardize prefixes
    if "production" in c:
        c = "cement_prod"
    elif "sales" in c:
        c = "cement_sales_local"
    elif "export" in c:
        c = "cement_exports"
    
    # clean characters
    c = re.sub(r"[^a-z0-9\s_-]", "", c)
    c = re.sub(r"[\s\-]+", "_", c)
    c = re.sub(r"_+", "_", c)
    c = c.strip("_")
    
    return c

df = df.rename(columns={c: clean_col(c) for c in df.columns})

df['date'] = pd.to_datetime(df['date'])

df.set_index("date", inplace=True)
df.sort_index(inplace=True)

df.dropna(axis=0, inplace=True, how='all')

print(f"Updated data range: {df.index.min()} to {df.index.max()}") 

df.to_csv(f"{PATH_RAW}/cbtt_cement.csv")

### Output - Production, Exports and Local Sales of Energy Commodities (Monthly)

In [ ]:
df = pd.read_csv(f"{PATH_RAW}/cbtt_prodoil.csv", parse_dates=True, index_col="date")
print(f"Previous data range: {df.index.min()} to {df.index.max()}") 


url = "https://www.central-bank.org.tt/data-center-block/?title=output-production-exports-and-local-sales-of-energy-commodities-monthly&id=149"

r = requests.get(url, headers=hdr, timeout=60)
r.raise_for_status()

soup = BeautifulSoup(r.text, "html.parser")

table = soup.select_one("table#table_1")
if table is None:
    raise ValueError("Could not find table_1 in the page HTML")

# headers
cols = [th.get_text(strip=True) for th in table.select("thead th")]

# rows (only real <tr> rows under tbody)
rows = []
for tr in table.select("tbody tr"):
    tds = [td.get_text(strip=True) for td in tr.select("td")]
    if tds:
        rows.append(tds)

df = pd.DataFrame(rows, columns=cols)

# clean types
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")

for c in df.columns[1:]:
    df[c] = pd.to_numeric(df[c].replace({"n.d.": np.nan, "n.a.": np.nan}), errors="coerce")

df = df.sort_values("Date").reset_index(drop=True)

def clean_col(c: str) -> str:
    c = c.lower()

    # remove units in parentheses
    c = re.sub(r"\s*\([^)]*\)", "", c)

    # standardize key commodities
    c = c.replace("crude oil", "oil")
    c = c.replace("natural gas", "gas")
    c = c.replace("liquefied natural gas", "lng")
    c = c.replace("motor gasolene", "motor_gasoline")
    c = c.replace("fuel oil", "fuel_oil")
    c = c.replace("natural gasoline", "natural_gasoline")
    c = c.replace("depth drilled", "drill_depth")
    c = c.replace("number of rig days", "rig_days")

    # standardize action
    if "daily average" in c:
        c = c.replace("production daily average", "prod_daily_avg")
    elif "production" in c:
        c = c.replace("production of ", "")
        c = c.replace("production ", "")
        c = c + "_prod"
    elif "exports" in c:
        c = c.replace("exports of ", "")
        c = c.replace("exports ", "")
        c = c + "_exports"
    elif "local sales" in c:
        c = c.replace("local sales of ", "")
        c = c.replace("local sales ", "")
        c = c + "_sales_local"

    # clean characters
    c = re.sub(r"[^a-z0-9\s_-]", "", c)
    c = re.sub(r"[\s\-]+", "_", c)
    c = re.sub(r"_+", "_", c)
    c = c.strip("_")

    return c


df = df.rename(columns={c: clean_col(c) for c in df.columns})

df['date'] = pd.to_datetime(df['date'])

df.set_index("date", inplace=True)
df.sort_index(inplace=True)

df.dropna(axis=0, inplace=True, how='all')

print(f"Updated data range: {df.index.min()} to {df.index.max()}") 

df.to_csv(f"{PATH_RAW}/cbtt_prodoil.csv")


### Prod. Index QIEA

In [ ]:
df = pd.read_csv(f"{PATH_RAW}/cbtt_qiea.csv", parse_dates=True, index_col="date")
print(f"Previous data range: {df.index.min()} to {df.index.max()}") 


url = "https://www.central-bank.org.tt/data-center-block/?title=output-quarterly-index-of-real-economic-activity-2012-100-index-values-quarterly&id=159"

r = requests.get(url, headers=hdr, timeout=60)
r.raise_for_status()

soup = BeautifulSoup(r.text, "html.parser")

table = soup.select_one("table#table_1")
if table is None:
    raise ValueError("Could not find table_1 in the page HTML")

# headers
cols = [th.get_text(strip=True) for th in table.select("thead th")]

# rows (only real <tr> rows under tbody)
rows = []
for tr in table.select("tbody tr"):
    tds = [td.get_text(strip=True) for td in tr.select("td")]
    if tds:
        rows.append(tds)

df = pd.DataFrame(rows, columns=cols)

# clean types
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")

for c in df.columns[1:]:
    df[c] = pd.to_numeric(df[c].replace({"n.d.": np.nan, "n.a.": np.nan}), errors="coerce")

df = df.sort_values("Date").reset_index(drop=True)
ABBR = {
    "mining_quarrying": "mining_quarry",
    "electricity_gas": "elec_gas",
    "water_supply": "water",
    "wholesale_retail_trade": "whl_rtl",
    "transportation_storage": "trans_storage",
    "accommodation_food_service_activities": "accom_food",
    "information_communication": "info_comm",
    "financial_insurance_activities": "fin_ins",
    "real_estate_activities": "real_est",
    "public_administration_defence": "pub_admin_def",
    "other_service_activities": "other_serv",
    "exploration_production": "expl_prod",
    "service_contractors": "svc_contract",
    "marketing_distribution": "mkt_dist",
    "electricity_water": "elec_water",
    # keep these mostly as-is (already short/clear)
    "manufacturing": "manufacturing",
    "construction": "construction",
    "agriculture": "agriculture",
    "education": "education",
    "energy": "energy",
    "refining": "refining",
    "petrochemicals": "petrochemicals",
    "non_energy": "non_energy",
}

def clean_col(c: str) -> str:
    c = c.lower()

    # extract parentheses (keep as suffix)
    parens = re.findall(r"\(([^)]*)\)", c)
    c = re.sub(r"\s*\([^)]*\)", "", c)

    # prefix + spelling
    c = c.replace("qiea index", "qiea")
    c = c.replace("accomodation", "accommodation")

    # separators
    c = c.replace(" - ", "_").replace(" and ", "_").replace("/", "_")

    # clean
    c = re.sub(r"[^a-z0-9\s_-]", "", c)
    c = re.sub(r"[\s\-]+", "_", c)
    c = re.sub(r"_+", "_", c).strip("_")

    # abbreviate main body (after qiea_)
    parts = c.split("_")
    if parts and parts[0] == "qiea":
        main = "_".join(parts[1:])
        main = ABBR.get(main, main)
        c = "qiea" if main == "" else f"qiea_{main}"

    # suffix from parentheses
    suffix = ""
    if parens:
        p = " ".join(parens).lower()

        # Normalize "Exc." to "ex" (and avoid excludingluding bug)
        p = re.sub(r"\bexc\.?\b", "ex", p)

        # normalize connectors
        p = p.replace("&", "and").replace("/", " ").replace("-", " ")

        # clean into tokens
        p = re.sub(r"[^a-z0-9\s]", " ", p)
        p = re.sub(r"\s+", "_", p).strip("_")

        # optional: shorten a couple common long suffix bits, but keep 'energy' as-is
        p = p.replace("petrochemicals", "petrochemicals")  # keep
        p = p.replace("refining", "refining")              # keep

        suffix = "_" + p if p else ""

    return c + suffix
df = df.rename(columns={c: clean_col(c) for c in df.columns})
df.columns= df.columns.astype(str).str.replace("_2012_100", "")

df['date'] = pd.to_datetime(df['date'])

df.set_index("date", inplace=True)
df.sort_index(inplace=True)

df.dropna(axis=0, inplace=True, how='all')

print(f"Updated data range: {df.index.min()} to {df.index.max()}") 

df.to_csv(f"{PATH_RAW}/cbtt_qiea.csv")


## Payment System

In [ ]:
df = pd.read_csv(f"{PATH_RAW}/cbtt_cc_transactions.csv", parse_dates=True, index_col="date")
print(f"Previous data range: {df.index.min()} to {df.index.max()}") 

url = "https://www.central-bank.org.tt/data-center-block?title=credit-card-transactions-monthly&id=110"
r = requests.get(url, headers=hdr, timeout=60)
r.raise_for_status()

soup = BeautifulSoup(r.text, "html.parser")

table = soup.select_one("table#table_1")
if table is None:
    raise ValueError("Could not find table_1 in the page HTML")

# headers
cols = [th.get_text(strip=True) for th in table.select("thead th")]

# rows (only real <tr> rows under tbody)
rows = []
for tr in table.select("tbody tr"):
    tds = [td.get_text(strip=True) for td in tr.select("td")]
    if tds:
        rows.append(tds)

df = pd.DataFrame(rows, columns=cols)

# clean types
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")

for c in df.columns[1:]:
    df[c] = pd.to_numeric(df[c].replace({"n.d.": np.nan, "n.a.": np.nan}), errors="coerce")

df = df.sort_values("Date").reset_index(drop=True)


df = df.rename(columns={
    "Date": "date",
    "Total Number of Credit Cards in Use": "cc_in_use",
    "Total Volume of Credit Card Automated Teller Machines (ATM) Withdrawals": "cc_atm_withdrawals_vol",
    "Total Value of Credit Card Automated Teller Machines (ATM) Withdrawals": "cc_atm_withdrawals_val",
    "Total Volume of Credit Card Point of Sale (POS) Transactions": "cc_pos_trans_vol",
    "Total Value of Credit Card Point of Sale (POS) Transactions": "cc_pos_trans_val",
})


df['date'] = pd.to_datetime(df['date'])

df.set_index("date", inplace=True)
df.sort_index(inplace=True)

df.dropna(axis=0, inplace=True, how='all')

print(f"Updated data range: {df.index.min()} to {df.index.max()}") 

df.to_csv(f"{PATH_RAW}/cbtt_cc_transactions.csv")

In [ ]:
df = pd.read_csv(f"{PATH_RAW}/cbtt_dc_transactions.csv", parse_dates=True, index_col="date")
print(f"Previous data range: {df.index.min()} to {df.index.max()}") 

url = "https://www.central-bank.org.tt/data-center-block/?title=debit-card-transactions-monthly&id=111"

r = requests.get(url, headers=hdr, timeout=60)
r.raise_for_status()

soup = BeautifulSoup(r.text, "html.parser")

table = soup.select_one("table#table_1")
if table is None:
    raise ValueError("Could not find table_1 in the page HTML")

# headers
cols = [th.get_text(strip=True) for th in table.select("thead th")]

# rows (only real <tr> rows under tbody)
rows = []
for tr in table.select("tbody tr"):
    tds = [td.get_text(strip=True) for td in tr.select("td")]
    if tds:
        rows.append(tds)

df = pd.DataFrame(rows, columns=cols)

# clean types
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")

for c in df.columns[1:]:
    df[c] = pd.to_numeric(df[c].replace({"n.d.": np.nan, "n.a.": np.nan}), errors="coerce")

df = df.sort_values("Date").reset_index(drop=True)


def clean_col(c: str) -> str:
    c0 = c.lower()

    # detect whether this column is volume/value/number (before stripping)
    is_vol = "total volume of" in c0
    is_val = "total value of" in c0
    is_num = "total number of" in c0

    # remove parentheses content (keeps "co-branded (with linx)" workable via "co-branded")
    c = re.sub(r"\s*\([^)]*\)", "", c0)

    # normalize key phrases
    c = c.replace("automated teller machines", "atm")
    c = c.replace("point of sale", "pos")

    # drop the "total ..." lead-in
    c = c.replace("total number of ", "")
    c = c.replace("total volume of ", "")
    c = c.replace("total value of ", "")

    # in-use types
    if "co-branded" in c and "debit cards in use" in c:
        return "dc_linx_in_use"   # co-branded with LINX
    if "internationally branded debit cards in use" in c:
        return "dc_intl_in_use"
    if "prepaid debit cards in use" in c:
        return "dc_prepaid_in_use"

    # ATM withdrawals
    if "debit card atm withdrawals" in c:
        if is_vol:
            return "dc_atm_withdrawals_vol"
        if is_val:
            return "dc_atm_withdrawals_val"
        return "dc_atm_withdrawals"

    # POS transactions
    if "debit card pos transactions" in c:
        if is_vol:
            return "dc_pos_trans_vol"
        if is_val:
            return "dc_pos_trans_val"
        return "dc_pos_trans"

    # fallback cleaner
    c = re.sub(r"[^a-z0-9\s_-]", "", c)
    c = re.sub(r"[\s\-]+", "_", c)
    c = re.sub(r"_+", "_", c).strip("_")
    return c

df = df.rename(columns={c: clean_col(c) for c in df.columns})


df['date'] = pd.to_datetime(df['date'])

df.set_index("date", inplace=True)
df.sort_index(inplace=True)

df.dropna(axis=0, inplace=True, how='all')

print(f"Updated data range: {df.index.min()} to {df.index.max()}") 
df.to_csv(f"{PATH_RAW}/cbtt_dc_transactions.csv")

In [ ]:
df = pd.read_csv(f"{PATH_RAW}/cbtt_pay_terminals.csv", parse_dates=True, index_col="date")
print(f"Previous data range: {df.index.min()} to {df.index.max()}") 

url = "https://www.central-bank.org.tt/data-center-block/?title=number-of-automated-teller-machines-atm-and-point-of-sales-pos-terminals-monthly&id=109"

r = requests.get(url, headers=hdr, timeout=60)
r.raise_for_status()

soup = BeautifulSoup(r.text, "html.parser")

table = soup.select_one("table#table_1")
if table is None:
    raise ValueError("Could not find table_1 in the page HTML")

# headers
cols = [th.get_text(strip=True) for th in table.select("thead th")]

# rows (only real <tr> rows under tbody)
rows = []
for tr in table.select("tbody tr"):
    tds = [td.get_text(strip=True) for td in tr.select("td")]
    if tds:
        rows.append(tds)

df = pd.DataFrame(rows, columns=cols)

# clean types
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")

for c in df.columns[1:]:
    df[c] = pd.to_numeric(df[c].replace({"n.d.": np.nan, "n.a.": np.nan}), errors="coerce")

df = df.sort_values("Date").reset_index(drop=True)

def clean_col(c: str) -> str:
    c = c.lower()
    
    # remove parentheses
    c = re.sub(r"\s*\([^)]*\)", "", c)
    
    # remove redundant phrases
    c = c.replace("total number of ", "")
    c = c.replace("automated teller machines", "atm")
    c = c.replace("point of sale", "pos")
    c = c.replace(" terminals", "_terminals")
    
    # clean characters
    c = re.sub(r"[^a-z0-9\s_-]", "", c)
    c = re.sub(r"[\s\-]+", "_", c)
    c = re.sub(r"_+", "_", c)
    c = c.strip("_")
    
    return c

df = df.rename(columns={c: clean_col(c) for c in df.columns})

df['date'] = pd.to_datetime(df['date'])

df.set_index("date", inplace=True)
df.sort_index(inplace=True)

df.dropna(axis=0, inplace=True, how='all')

print(f"Updated data range: {df.index.min()} to {df.index.max()}") 

df.to_csv(f"{PATH_RAW}/cbtt_pay_terminals.csv")

## Foreign Reserves

In [ ]:
df = pd.read_csv(f"{PATH_RAW}/cbtt_fxres.csv", parse_dates=True, index_col="date")
print(f"Previous data range: {df.index.min()} to {df.index.max()}") 


url = "https://www.central-bank.org.tt/data-center-block/?title=foreign-reserves-monthly&id=119"

r = requests.get(url, headers=hdr, timeout=60)
r.raise_for_status()

soup = BeautifulSoup(r.text, "html.parser")

table = soup.select_one("table#table_1")
if table is None:
    raise ValueError("Could not find table_1 in the page HTML")

# headers
cols = [th.get_text(strip=True) for th in table.select("thead th")]

# rows (only real <tr> rows under tbody)
rows = []
for tr in table.select("tbody tr"):
    tds = [td.get_text(strip=True) for td in tr.select("td")]
    if tds:
        rows.append(tds)

df = pd.DataFrame(rows, columns=cols)

# clean types
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")

for c in df.columns[1:]:
    df[c] = pd.to_numeric(df[c].replace({"n.d.": np.nan, "n.a.": np.nan}), errors="coerce")

df = df.sort_values("Date").reset_index(drop=True)

rename_map = {
    "Date": "date",
    "T&T Gross Foreign Assets - US$ Millions": "tt_gross_foreign_assets",
    "T&T Total Foreign Liabilities - US$ Millions": "tt_total_foreign_liabilities",
    "T&T Net Foreign Position - US$ Millions": "tt_net_foreign_position",
    "Net Official Reserves - US$ Millions": "net_official_reserves",
    "Import Cover": "import_cover"
}

df = df.rename(columns=rename_map)


df['date'] = pd.to_datetime(df['date'])

df.set_index("date", inplace=True)
df.sort_index(inplace=True)

df.dropna(axis=0, inplace=True, how='all')

print(f"Updated data range: {df.index.min()} to {df.index.max()}") 

df.to_csv(f"{PATH_RAW}/cbtt_fxres.csv")

## Labour Force

In [ ]:
df = pd.read_csv(f"{PATH_RAW}/cbtt_lab_ur.csv", parse_dates=True, index_col="date")
print(f"Previous data range: {df.index.min()} to {df.index.max()}") 


url = "https://www.central-bank.org.tt/data-center-block/?title=labour-force-unemployment-participation-rates-quarterly&id=122"

r = requests.get(url, headers=hdr, timeout=60)
r.raise_for_status()

soup = BeautifulSoup(r.text, "html.parser")

table = soup.select_one("table#table_1")
if table is None:
    raise ValueError("Could not find table_1 in the page HTML")

# headers
cols = [th.get_text(strip=True) for th in table.select("thead th")]

# rows (only real <tr> rows under tbody)
rows = []
for tr in table.select("tbody tr"):
    tds = [td.get_text(strip=True) for td in tr.select("td")]
    if tds:
        rows.append(tds)

df = pd.DataFrame(rows, columns=cols)

# clean types
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")

for c in df.columns[1:]:
    df[c] = pd.to_numeric(df[c].replace({"n.d.": np.nan, "n.a.": np.nan}), errors="coerce")

df = df.sort_values("Date").reset_index(drop=True)

def clean_col(c: str) -> str:
    c = c.lower()
    
    # remove units
    c = re.sub(r"\s*\([^)]*\)", "", c)
    
    # standardize wording
    c = c.replace("unemployment rate", "unemp_rate")
    c = c.replace("labour force participation rate", "lfp_rate")
    
    # clean characters
    c = re.sub(r"[^a-z0-9\s_-]", "", c)
    c = re.sub(r"[\s\-]+", "_", c)
    c = re.sub(r"_+", "_", c)
    c = c.strip("_")
    
    return c

df = df.rename(columns={c: clean_col(c) for c in df.columns})

df['date'] = pd.to_datetime(df['date'])

df.set_index("date", inplace=True)
df.sort_index(inplace=True)

df.dropna(axis=0, inplace=True, how='all')

print(f"Updated data range: {df.index.min()} to {df.index.max()}") 

df.to_csv(f"{PATH_RAW}/cbtt_lab_ur.csv")

In [ ]:
df = pd.read_csv(f"{PATH_RAW}/cbtt_lab_lf.csv", parse_dates=True, index_col="date")
print(f"Previous data range: {df.index.min()} to {df.index.max()}") 


url = "https://www.central-bank.org.tt/data-center-block/?title=labour-force-quarterly&id=198"


r = requests.get(url, headers=hdr, timeout=60)
r.raise_for_status()

soup = BeautifulSoup(r.text, "html.parser")

table = soup.select_one("table#table_1")
if table is None:
    raise ValueError("Could not find table_1 in the page HTML")

# headers
cols = [th.get_text(strip=True) for th in table.select("thead th")]

# rows (only real <tr> rows under tbody)
rows = []
for tr in table.select("tbody tr"):
    tds = [td.get_text(strip=True) for td in tr.select("td")]
    if tds:
        rows.append(tds)

df = pd.DataFrame(rows, columns=cols)

# clean types
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")

for c in df.columns[1:]:
    df[c] = pd.to_numeric(df[c].replace({"n.d.": np.nan, "n.a.": np.nan}), errors="coerce")

df = df.sort_values("Date").reset_index(drop=True)

def clean_col(c: str) -> str:
    c = c.lower()
    
    # remove units in parentheses
    c = re.sub(r"\s*\([^)]*\)", "", c)
    
    # standardize wording
    c = c.replace("total ", "")
    c = c.replace("non-institutional ", "")
    c = c.replace(" years old and over", "")
    
    # shorten common terms
    c = c.replace("employment", "emp")
    c = c.replace("labour force", "lf")
    c = c.replace("unemployment", "unemp")
    c = c.replace("population 15", "pop_15plus")
    
    # clean characters
    c = re.sub(r"[^a-z0-9\s_-]", "", c)
    c = re.sub(r"[\s\-]+", "_", c)
    c = re.sub(r"_+", "_", c)
    c = c.strip("_")
    
    return c

df = df.rename(columns={c: clean_col(c) for c in df.columns})

df['date'] = pd.to_datetime(df['date'])

df.set_index("date", inplace=True)
df.sort_index(inplace=True)

df.dropna(axis=0, inplace=True, how='all')

print(f"Updated data range: {df.index.min()} to {df.index.max()}") 

df.to_csv(f"{PATH_RAW}/cbtt_lab_lf.csv")


In [ ]:
df = pd.read_csv(f"{PATH_RAW}/cbtt_lab_sec.csv", parse_dates=True, index_col="date")
print(f"Previous data range: {df.index.min()} to {df.index.max()}") 

url = "https://www.central-bank.org.tt/data-center-block/?title=labour-force-employment-by-industry-quarterly&id=196"

r = requests.get(url, headers=hdr, timeout=60)
r.raise_for_status()

soup = BeautifulSoup(r.text, "html.parser")

table = soup.select_one("table#table_1")
if table is None:
    raise ValueError("Could not find table_1 in the page HTML")

# headers
cols = [th.get_text(strip=True) for th in table.select("thead th")]

# rows (only real <tr> rows under tbody)
rows = []
for tr in table.select("tbody tr"):
    tds = [td.get_text(strip=True) for td in tr.select("td")]
    if tds:
        rows.append(tds)

df = pd.DataFrame(rows, columns=cols)

# clean types
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")

for c in df.columns[1:]:
    df[c] = pd.to_numeric(df[c].replace({"n.d.": np.nan, "n.a.": np.nan}), errors="coerce")

df = df.sort_values("Date").reset_index(drop=True)

def clean_col(c: str) -> str:
    c = c.lower()
    
    # remove units
    c = re.sub(r"\s*\([^)]*\)", "", c)
    
    # standardize prefix
    c = c.replace("employment in ", "emp_")
    c = c.replace("employment - ", "emp_")
    
    # simplify sector names
    c = c.replace(" (incl. electricity & water)", "")
    c = c.replace("electricity & water", "electricity_water")
    c = c.replace("petroleum and gas", "petroleum_gas")
    c = c.replace("transport, storage and communication", "transport_storage_comm")
    c = c.replace("all other sectors", "other_sectors")
    c = c.replace("sector not stated", "sector_not_stated")
    
    # clean characters
    c = re.sub(r"[^a-z0-9\s_-]", "", c)
    c = re.sub(r"[\s\-&,]+", "_", c)
    c = re.sub(r"_+", "_", c)
    c = c.strip("_")
    
    return c

df = df.rename(columns={c: clean_col(c) for c in df.columns})
df['date'] = pd.to_datetime(df['date'])

df.set_index("date", inplace=True)
df.sort_index(inplace=True)

df.dropna(axis=0, inplace=True, how='all')

print(f"Updated data range: {df.index.min()} to {df.index.max()}") 

df.to_csv(f"{PATH_RAW}/cbtt_lab_sec.csv")

## Liquidity

In [ ]:
df = pd.read_csv(f"{PATH_RAW}/cbtt_liq.csv", parse_dates=True, index_col="date")
print(f"Previous data range: {df.index.min()} to {df.index.max()}") 

url = "https://www.central-bank.org.tt/data-center-block/?title=liquidity-monthly&id=121"


r = requests.get(url, headers=hdr, timeout=60)
r.raise_for_status()

soup = BeautifulSoup(r.text, "html.parser")

table = soup.select_one("table#table_1")
if table is None:
    raise ValueError("Could not find table_1 in the page HTML")

# headers
cols = [th.get_text(strip=True) for th in table.select("thead th")]

# rows (only real <tr> rows under tbody)
rows = []
for tr in table.select("tbody tr"):
    tds = [td.get_text(strip=True) for td in tr.select("td")]
    if tds:
        rows.append(tds)

df = pd.DataFrame(rows, columns=cols)

# clean types
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")

for c in df.columns[1:]:
    df[c] = pd.to_numeric(df[c].replace({"n.d.": np.nan, "n.a.": np.nan}), errors="coerce")

df = df.sort_values("Date").reset_index(drop=True)

def clean_col(c: str) -> str:
    c = c.lower()
    
    # remove units
    c = re.sub(r"\s*\([^)]*\)", "", c)
    
    # standardize institutional prefixes
    c = c.replace("commercial banks ", "cbk_")
    c = c.replace("commercial bank ", "cbk_")
    c = c.replace("central government ", "cg_")
    
    # simplify terms
    c = c.replace("average excess reserves", "avg_excess_reserves")
    c = c.replace("net domestic fiscal injections", "net_dom_fiscal_inj")
    c = c.replace("net open market operations", "net_omo")
    c = c.replace("required reserves", "req_reserves")
    c = c.replace("cash reserves", "cash_reserves")
    c = c.replace("excess reserves-end of period", "excess_reserves_eop")
    c = c.replace("prescribed deposits liabilities", "prescribed_dep_liab")
    c = c.replace("deposits - secondary reserve requirements, fixed deposits & special deposits",
                  "sec_reserve_req_fixed_special_dep")
    c = c.replace("treasury bills", "t_bills")
    c = c.replace("local cash in hand", "cash_in_hand")
    
    # clean characters
    c = re.sub(r"[^a-z0-9\s_-]", "", c)
    c = re.sub(r"[\s\-&,]+", "_", c)
    c = re.sub(r"_+", "_", c)
    c = c.strip("_")
    
    return c

df = df.rename(columns={c: clean_col(c) for c in df.columns})


df['date'] = pd.to_datetime(df['date'])

df.set_index("date", inplace=True)
df.sort_index(inplace=True)

df.dropna(axis=0, inplace=True, how='all')

print(f"Updated data range: {df.index.min()} to {df.index.max()}") 

df.to_csv(f"{PATH_RAW}/cbtt_liq.csv")

## External Sector - Balance of Payments (Quarterly)

In [ ]:
df = pd.read_csv(f"{PATH_RAW}/cbtt_bop.csv", parse_dates=True, index_col="date")
print(f"Previous data range: {df.index.min()} to {df.index.max()}") 

url = "https://www.central-bank.org.tt/data-center-block/?title=external-sector-balance-of-payments-quarterly&id=124"

r = requests.get(url, headers=hdr, timeout=60)
r.raise_for_status()

soup = BeautifulSoup(r.text, "html.parser")

table = soup.select_one("table#table_1")
if table is None:
    raise ValueError("Could not find table_1 in the page HTML")

# headers
cols = [th.get_text(strip=True) for th in table.select("thead th")]

# rows (only real <tr> rows under tbody)
rows = []
for tr in table.select("tbody tr"):
    tds = [td.get_text(strip=True) for td in tr.select("td")]
    if tds:
        rows.append(tds)

df = pd.DataFrame(rows, columns=cols)

# clean types
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")

for c in df.columns[1:]:
    df[c] = pd.to_numeric(df[c].replace({"n.d.": np.nan, "n.a.": np.nan}), errors="coerce")

df = df.sort_values("Date").reset_index(drop=True)

rename_map = {
    "Date": "date",

    "Current Account (US$Mn)": "bop_current_account",
    "Goods and Services (US$Mn)": "bop_goods_services",
    "Goods, net (US$Mn)": "bop_goods_net",
    "Exports (US$Mn)": "bop_exports",
    "Energy Exports (US$Mn)": "bop_exports_energy",
    "Non-energy Exports (US$Mn)": "bop_exports_nonenergy",
    "Imports (US$Mn)": "bop_imports",
    "Fuel Imports (US$Mn)": "bop_imports_fuel",
    "Other Imports (US$Mn)": "bop_imports_other",

    "Services, net (US$Mn)": "bop_services_net",
    "Services Receipts (US$Mn)": "bop_services_receipts",
    "Services Payments (US$Mn)": "bop_services_payments",

    "Primary Income, net (US$Mn)": "bop_primary_income_net",
    "Primary Income Receipts (US$Mn)": "bop_primary_income_receipts",
    "Primary Income Payments (US$Mn)": "bop_primary_income_payments",

    "Secondary Income, net (US$Mn)": "bop_secondary_income_net",
    "Secondary Income Receipts (US$Mn)": "bop_secondary_income_receipts",
    "Secondary Income Receipts:     Government transfers (US$Mn)": "bop_secondary_income_receipts_gov",
    "Secondary Income Receipts:      Private transfers (US$Mn)": "bop_secondary_income_receipts_private",
    "Secondary Income Payments (US$Mn)": "bop_secondary_income_payments",
    "Secondary Income Payments:     Government transfers (US$Mn)": "bop_secondary_income_payments_gov",
    "Secondary Income Payments:      Private transfers (US$Mn)": "bop_secondary_income_payments_private",

    "Capital Account (US$Mn)": "bop_capital_account",
    "Financial Account (US$Mn)": "bop_financial_account",

    "Direct Investment (US$Mn)": "bop_direct_investment",
    "Direct Investment: Net Acquisition of Financial Assets (US$Mn)": "bop_direct_inv_assets",
    "Direct Investment: Net Incurrence of Financial Liabilities (US$Mn)": "bop_direct_inv_liabilities",

    "Portfolio Investment (US$Mn)": "bop_portfolio_investment",
    "Portfolio Investment: Net Acquisition of Financial Assets (US$Mn)": "bop_portfolio_assets",
    "Portfolio Investment: Net Incurrence of Financial Liabilities (US$Mn)": "bop_portfolio_liabilities",

    "Financial Derivatives (US$Mn)": "bop_fin_derivatives",
    "Financial Derivatives: Net Acquisition of Financial Assets (US$Mn)": "bop_fin_deriv_assets",
    "Financial Derivatives: Net Incurrence of Financial Liabilities (US$Mn)": "bop_fin_deriv_liabilities",

    "Other Investment (US$Mn)": "bop_other_investment",
    "Other Investment: Net Acquisition of Financial Assets (US$Mn)": "bop_other_inv_assets",
    "Other Investment: Net Incurrence of Financial Liabilities (US$Mn)": "bop_other_inv_liabilities",

    "Net Errors and Omissions (US$Mn)": "bop_net_errors_omissions",
    "Overall Balance (US$Mn)": "bop_overall_balance",
}

df = df.rename(columns=rename_map)

df['date'] = pd.to_datetime(df['date'])

df.set_index("date", inplace=True)
df.sort_index(inplace=True)

df.dropna(axis=0, inplace=True, how='all')

print(f"Updated data range: {df.index.min()} to {df.index.max()}") 

df.to_csv(f"{PATH_RAW}/cbtt_bop.csv")

## Debt

In [ ]:
df = pd.read_csv(f"{PATH_RAW}/cbtt_debt_cg.csv", parse_dates=True, index_col="date")
print(f"Previous data range: {df.index.min()} to {df.index.max()}") 

url = "https://www.central-bank.org.tt/data-center-block/?title=debt-central-government-monthly&id=172"

r = requests.get(url, headers=hdr, timeout=60)
r.raise_for_status()

soup = BeautifulSoup(r.text, "html.parser")

table = soup.select_one("table#table_1")
if table is None:
    raise ValueError("Could not find table_1 in the page HTML")

# headers
cols = [th.get_text(strip=True) for th in table.select("thead th")]

# rows (only real <tr> rows under tbody)
rows = []
for tr in table.select("tbody tr"):
    tds = [td.get_text(strip=True) for td in tr.select("td")]
    if tds:
        rows.append(tds)

df = pd.DataFrame(rows, columns=cols)

# clean types
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")

for c in df.columns[1:]:
    df[c] = pd.to_numeric(df[c].replace({"n.d.": np.nan, "n.a.": np.nan}), errors="coerce")

df = df.sort_values("Date").reset_index(drop=True)

def clean_col(c: str) -> str:
    c = c.lower()
    
    # remove units
    c = re.sub(r"\s*\([^)]*\)", "", c)
    
    # standardize prefix
    c = c.replace("central government ", "cg_")
    
    # shorten key instruments
    replacements = {
        "debt management bills": "dmb",
        "open market treasury note": "omt_note",
        "open market treasury notes": "omt_notes",
        "liquidity absorption bonds": "lab",
        "cliko and hcu zero-coupon bonds": "clico_hcu_zcb",
        "cliko and hcu zero coupon bonds": "clico_hcu_zcb",
        "clico and hcu zero-coupon bonds": "clico_hcu_zcb",
        "domestic bonds and notes": "dom_bonds_notes",
        "domestic bonds & notes": "dom_bonds_notes",
        "domestic bond issuances for vat refunds": "vat_refund_bonds_iss",
        "domestic repayments on vat refund bonds": "vat_refund_bonds_repay",
        "domestic vat refund bonds": "vat_refund_bonds",

        "domestic debt": "dom_debt",
        "external debt": "ext_debt",
        "total debt": "total_debt",
        "total sterilized debt": "total_sterilized_debt",
    }
    
    for k, v in replacements.items():
        c = c.replace(k, v)
    
    # standardize actions
    c = c.replace(" issuances", "_iss")
    c = c.replace(" issues", "_iss")
    c = c.replace(" repayments", "_repay")
    c = c.replace(" conversions and redemptions", "_conv_redeem")
    c = c.replace(" allocations", "_alloc")
    c = c.replace(" outstanding", "_outstanding")
    
    # clean characters
    c = re.sub(r"[^a-z0-9\s_-]", "", c)
    c = re.sub(r"[\s\-]+", "_", c)
    c = re.sub(r"_+", "_", c)
    c = c.strip("_")
    
    return c


df = df.rename(columns={c: clean_col(c) for c in df.columns})


df['date'] = pd.to_datetime(df['date'])

df.set_index("date", inplace=True)
df.sort_index(inplace=True)

df.dropna(axis=0, inplace=True, how='all')

print(f"Updated data range: {df.index.min()} to {df.index.max()}") 

df.to_csv(f"{PATH_RAW}/cbtt_debt_cg.csv")

In [ ]:
df = pd.read_csv(f"{PATH_RAW}/cbtt_debt_gg.csv", parse_dates=True, index_col="date")
print(f"Previous data range: {df.index.min()} to {df.index.max()}") 

url = "https://www.central-bank.org.tt/data-center-block/?title=debt-general-government-quarterly&id=177"

r = requests.get(url, headers=hdr, timeout=60)
r.raise_for_status()

soup = BeautifulSoup(r.text, "html.parser")

table = soup.select_one("table#table_1")
if table is None:
    raise ValueError("Could not find table_1 in the page HTML")

# headers
cols = [th.get_text(strip=True) for th in table.select("thead th")]

# rows (only real <tr> rows under tbody)
rows = []
for tr in table.select("tbody tr"):
    tds = [td.get_text(strip=True) for td in tr.select("td")]
    if tds:
        rows.append(tds)

df = pd.DataFrame(rows, columns=cols)

# clean types
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")

for c in df.columns[1:]:
    df[c] = pd.to_numeric(df[c].replace({"n.d.": np.nan, "n.a.": np.nan}), errors="coerce")

df = df.sort_values("Date").reset_index(drop=True)

def clean_col(c: str) -> str:
    c = c.lower()

    # Remove units
    c = re.sub(r"\s*\([^)]*\)", "", c)

    # Standardize government prefixes
    c = c.replace("general government", "gg")
    c = c.replace("adjusted general government", "gg_adj")
    c = c.replace("central government", "cg")

    # Standardize phrases
    c = c.replace("non self-serviced", "nonself")
    c = c.replace("non-self serviced", "nonself")
    c = c.replace("self-serviced", "self")
    c = c.replace("state-owned enterprise", "soe")
    c = c.replace("statutory bodies", "stat_bodies")
    c = c.replace("guaranteed debt", "guar_debt")
    c = c.replace("debt outstanding", "debt_outstanding")
    c = c.replace("debt to gdp ratio", "debt_gdp_ratio")
    c = c.replace("debt service ratio", "debt_service_ratio")

    # Clean characters
    c = re.sub(r"[^a-z0-9\s_-]", "", c)
    c = re.sub(r"[\s\-]+", "_", c)
    c = re.sub(r"_+", "_", c)
    c = c.strip("_")

    return c

df = df.rename(columns={c: clean_col(c) for c in df.columns})

df['date'] = pd.to_datetime(df['date'])

df.set_index("date", inplace=True)
df.sort_index(inplace=True)

df.dropna(axis=0, inplace=True, how='all')

print(f"Updated data range: {df.index.min()} to {df.index.max()}") 

df.to_csv(f"{PATH_RAW}/cbtt_debt_gg.csv")


## Capital Market

In [ ]:
df = pd.read_csv(f"{PATH_RAW}/cbtt_capmark.csv", parse_dates=True, index_col="date")
print(f"Previous data range: {df.index.min()} to {df.index.max()}") 

url = "https://www.central-bank.org.tt/data-center-block/?title=capital-market-monthly&id=27"

r = requests.get(url, headers=hdr, timeout=60)
r.raise_for_status()

soup = BeautifulSoup(r.text, "html.parser")

table = soup.select_one("table#table_1")
if table is None:
    raise ValueError("Could not find table_1 in the page HTML")

# headers
cols = [th.get_text(strip=True) for th in table.select("thead th")]

# rows (only real <tr> rows under tbody)
rows = []
for tr in table.select("tbody tr"):
    tds = [td.get_text(strip=True) for td in tr.select("td")]
    if tds:
        rows.append(tds)

df = pd.DataFrame(rows, columns=cols)

# clean types
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")

for c in df.columns[1:]:
    df[c] = pd.to_numeric(df[c].replace({"n.d.": np.nan, "n.a.": np.nan}), errors="coerce")

df = df.sort_values("Date").reset_index(drop=True)

def clean_col(c: str) -> str:
    c = c.lower()
    
    # remove parentheses content
    c = re.sub(r"\s*\([^)]*\)", "", c)

    # standardize key phrases
    c = c.replace("composite stock price index", "stock_index")
    c = c.replace("market turnover value - ttse", "ttse_turnover_value")
    c = c.replace("number of transactions - ttse", "ttse_transactions")
    c = c.replace("volume of shares traded - ttse", "ttse_share_volume")
    
    c = c.replace("holdings of debt management bills", "dmb_holdings")
    c = c.replace("holdings of debt management-total holdings", "dmb_holdings_total")
    
    c = c.replace("holdings of open market bills and notes", "ombn_holdings")
    c = c.replace("grand total of the holdings of treasury bills", "treasury_bill_holdings_total")

    # institution standardization
    c = c.replace("central bank", "cb")
    c = c.replace("commercial banks", "comm_banks")
    c = c.replace("other", "other")
    c = c.replace("total holdings", "total")

    # general cleanup
    c = re.sub(r"[^a-z0-9\s_-]", "", c)
    c = re.sub(r"[\s\-]+", "_", c)
    c = re.sub(r"_+", "_", c)
    c = c.strip("_")
    
    return c

df = df.rename(columns={c: clean_col(c) for c in df.columns})

df['date'] = pd.to_datetime(df['date'])

df.set_index("date", inplace=True)
df.sort_index(inplace=True)

df.dropna(axis=0, inplace=True, how='all')

print(f"Updated data range: {df.index.min()} to {df.index.max()}") 

df.to_csv(f"{PATH_RAW}/cbtt_capmark.csv")

## Financial Soundness

In [ ]:
df = pd.read_csv(f"{PATH_RAW}/cbtt_findsound.csv", parse_dates=True, index_col="date")
print(f"Previous data range: {df.index.min()} to {df.index.max()}") 

url = "https://www.central-bank.org.tt/data-center-block/?title=financial-soundness-indicators-quarterly&id=89"

r = requests.get(url, headers=hdr, timeout=60)
r.raise_for_status()

soup = BeautifulSoup(r.text, "html.parser")

table = soup.select_one("table#table_1")
if table is None:
    raise ValueError("Could not find table_1 in the page HTML")

# headers
cols = [th.get_text(strip=True) for th in table.select("thead th")]

# rows (only real <tr> rows under tbody)
rows = []
for tr in table.select("tbody tr"):
    tds = [td.get_text(strip=True) for td in tr.select("td")]
    if tds:
        rows.append(tds)

df = pd.DataFrame(rows, columns=cols)

# clean types
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")

for c in df.columns[1:]:
    df[c] = pd.to_numeric(df[c].replace({"n.d.": np.nan, "n.a.": np.nan}), errors="coerce")

df = df.sort_values("Date").reset_index(drop=True)

def clean_col(c: str) -> str:
    c = c.lower()

    # remove units
    c = re.sub(r"\s*\([^)]*\)", "", c)

    # remove prefix
    c = c.replace("commercial banks:", "").strip()

    # standardize common terms
    c = c.replace("customer deposits-to-total loans", "cb_deposits_loans_ratio")
    c = c.replace("foreign currency liabilities-to-total liabilities", "cb_fx_liab_total_liab_ratio")
    c = c.replace("general provisions-to-gross loans", "cb_gen_prov_gross_loans_ratio")
    c = c.replace("specific provisions-to-gross loans", "cb_spec_prov_gross_loans_ratio")
    c = c.replace("interest margin-to-gross income", "cb_interest_margin_income_ratio")
    c = c.replace("liquid assets-to-total short-term liabilities", "cb_liquid_assets_st_liab_ratio")
    c = c.replace("net open position in foreign exchange-to-capital", "cb_fx_open_pos_cap_ratio")
    c = c.replace("non-interest expenses-to-gross income", "cb_nonint_exp_income_ratio")
    c = c.replace("non-interest income-to-gross income", "cb_nonint_inc_income_ratio")
    c = c.replace("non-performing loans-to-gross loans", "cb_npl_gross_loans_ratio")
    c = c.replace("non-performing loans net of provisions-to-capital", "cb_npl_net_cap_ratio")
    c = c.replace("regulatory capital-to-risk weighted assets", "cb_reg_cap_rwa_ratio")
    c = c.replace("regulatory capital-to-total assets", "cb_reg_cap_assets_ratio")
    c = c.replace("regulatory tier i capital-to-risk-weighted assets", "cb_tier1_rwa_ratio")
    c = c.replace("regulatory tier ii capital-to-risk-weighted assets", "cb_tier2_rwa_ratio")
    c = c.replace("return on assets", "cb_roa")
    c = c.replace("return on equity", "cb_roe")
    c = c.replace("specific provisions-to-impaired loans", "cb_spec_prov_impaired_ratio")
    c = c.replace("total provisions-to-impaired loans", "cb_total_prov_impaired_ratio")

    # final cleanup
    c = re.sub(r"[^a-z0-9\s_-]", "", c)
    c = re.sub(r"[\s\-]+", "_", c)
    c = re.sub(r"_+", "_", c)
    c = c.strip("_")

    return c

df = df.rename(columns={c: clean_col(c) for c in df.columns})


df['date'] = pd.to_datetime(df['date'])

df.set_index("date", inplace=True)
df.sort_index(inplace=True)

df.dropna(axis=0, inplace=True, how='all')

print(f"Updated data range: {df.index.min()} to {df.index.max()}") 

df.to_csv(f"{PATH_RAW}/cbtt_findsound.csv")

## Commercial Banks (Monthly)

In [ ]:
try:
    df = pd.read_csv(f"{PATH_RAW}/cbtt_cb_credit.csv", parse_dates=True, index_col="date")
    print(f"Previous data range: {df.index.min()} to {df.index.max()}") 
except:
    print(f"No previous data found for cbtt_cb_credit.csv") 

url = "https://www.central-bank.org.tt/data-center-block/?title=commercial-banks-credit-monthly&id=72"

r = requests.get(url, headers=hdr, timeout=60)
r.raise_for_status()

soup = BeautifulSoup(r.text, "html.parser")

table = soup.select_one("table#table_1")
if table is None:
    raise ValueError("Could not find table_1 in the page HTML")

# headers
cols = [th.get_text(strip=True) for th in table.select("thead th")]

# rows (only real <tr> rows under tbody)
rows = []
for tr in table.select("tbody tr"):
    tds = [td.get_text(strip=True) for td in tr.select("td")]
    if tds:
        rows.append(tds)

df = pd.DataFrame(rows, columns=cols)

# clean types
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")
df = df.sort_values("Date").reset_index(drop=True)
df.columns = df.columns.astype(str).str.lower()
df.set_index("date", inplace=True)
df.index = pd.to_datetime(df.index)
df.sort_index(inplace=True)

for c in df.columns:
    df[c] = pd.to_numeric(df[c].replace({"n.d.": np.nan, "n.a.": np.nan}), errors="coerce")

def clean_col(c: str) -> str:
    if c == "date":
        return c

    # tag y-o-y before stripping parenthetical content
    is_yoy = "y-o-y" in c

    # remove parenthetical content (units, qualifiers)
    c = re.sub(r"\s*\([^)]*\)", "", c)

    # strip institution prefix (with or without colon)
    c = re.sub(r"commercial banks:?\s*", "", c).strip()

    # standardize terms
    c = c.replace("private sector credit", "pvt_credit")
    c = c.replace("lending to business firms", "biz_credit")
    c = c.replace("loans to businesses", "biz_loans")
    c = c.replace("lending to consumers", "cons_credit")
    c = c.replace("real estate credit", "re_credit")
    c = c.replace("central government credit", "cg_credit")
    c = c.replace("claims against central government", "cg_claims")
    c = c.replace("other domestic credit", "other_dom_credit")
    c = c.replace("public sector credit", "pub_credit")

    # final cleanup
    c = re.sub(r"[^a-z0-9\s_-]", "", c)
    c = re.sub(r"[\s\-]+", "_", c)
    c = re.sub(r"_+", "_", c)
    c = c.strip("_")

    # add cbk_ prefix and optional _yoy suffix
    c = "cbk_" + c
    if is_yoy:
        c += "_yoy"

    return c

df = df.rename(columns={c: clean_col(c) for c in df.columns})

df.dropna(axis=0, inplace=True, how="all")

print(f"Updated data range: {df.index.min()} to {df.index.max()}")
df.to_csv(f"{PATH_RAW}/cbtt_cb_credit.csv")


In [ ]:
try:
    df = pd.read_csv(f"{PATH_RAW}/cbtt_cb_deposits.csv", parse_dates=True, index_col="date")
    print(f"Previous data range: {df.index.min()} to {df.index.max()}")
except:
    print(f"No previous data found for cbtt_cb_deposits.csv")

url = "https://www.central-bank.org.tt/data-center-block/?title=commercial-banks-deposits-monthly&id=75"

r = requests.get(url, headers=hdr, timeout=60)
r.raise_for_status()

soup = BeautifulSoup(r.text, "html.parser")

table = soup.select_one("table#table_1")
if table is None:
    raise ValueError("Could not find table_1 in the page HTML")

# headers
cols = [th.get_text(strip=True) for th in table.select("thead th")]

# rows (only real <tr> rows under tbody)
rows = []
for tr in table.select("tbody tr"):
    tds = [td.get_text(strip=True) for td in tr.select("td")]
    if tds:
        rows.append(tds)

df = pd.DataFrame(rows, columns=cols)

# clean types
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")
df = df.sort_values("Date").reset_index(drop=True)
df.columns = df.columns.astype(str).str.lower()
df.set_index("date", inplace=True)
df.index = pd.to_datetime(df.index)
df.sort_index(inplace=True)

for c in df.columns:
    df[c] = pd.to_numeric(df[c].replace({"n.d.": np.nan, "n.a.": np.nan}), errors="coerce")

def clean_col(c: str) -> str:
    if c == "date":
        return c

    # remove unit suffixes: parenthetical or dash-separated (tt/us millions)
    c = re.sub(r"\s*\([^)]*\)", "", c)
    c = re.sub(r"\s*-\s*(tt|us)\$\s*millions?", "", c)

    # strip institution prefix
    c = re.sub(r"commercial banks:\s*", "", c).strip()

    # time deposit maturity buckets (specific before generic)
    c = c.replace("16 days to 3 months deposits", "dep_16d_3m")
    c = c.replace("3 to 6 months deposits", "dep_3m_6m")
    c = c.replace("6 months to 1 year deposits", "dep_6m_1y")
    c = c.replace("over 1 year time deposits", "dep_time_1yplus")
    c = c.replace("call deposits", "dep_call")

    # demand deposits (specific before generic)
    c = c.replace("non-interest bearing demand deposits", "dep_demand_nonint")
    c = c.replace("interest bearing demand deposits", "dep_demand_int")
    c = c.replace("demand deposits", "dep_demand")

    # savings (specific before generic)
    c = c.replace("ordinary and cheque savings deposits", "dep_sav_ord_cheque")
    c = c.replace("savings deposits", "dep_sav")

    # time and special
    c = c.replace("special deposits", "dep_special")
    c = c.replace("time deposits", "dep_time")

    # foreign currency
    c = c.replace("foreign currency deposits", "dep_fx")

    # by depositor type (unincorporated before incorporated)
    c = c.replace("unincorporated businesses deposits", "dep_uninc_biz")
    c = c.replace("incorporated businesses deposits", "dep_inc_biz")
    c = c.replace("non-financial state enterprises deposits", "dep_nonfin_state_ent")
    c = c.replace("state-owned financial enterprises deposits", "dep_state_fin_ent")
    c = c.replace("private financial institutions deposits", "dep_pvt_fin_inst")
    c = c.replace("public sector deposits", "dep_pub")
    c = c.replace("total deposits", "dep_total")

    # final cleanup
    c = re.sub(r"[^a-z0-9\s_-]", "", c)
    c = re.sub(r"[\s\-]+", "_", c)
    c = re.sub(r"_+", "_", c)
    c = c.strip("_")

    # add cbk_ prefix
    c = "cbk_" + c

    return c

df = df.rename(columns={c: clean_col(c) for c in df.columns})

df.dropna(axis=0, inplace=True, how="all")

print(f"Updated data range: {df.index.min()} to {df.index.max()}")
df.to_csv(f"{PATH_RAW}/cbtt_cb_deposits.csv")

## Foreign Currency Assets And Liabilities (Monthly)

In [ ]:
try:
    df = pd.read_csv(f"{PATH_RAW}/cbtt_cb_fx.csv", parse_dates=True, index_col="date")
    print(f"Previous data range: {df.index.min()} to {df.index.max()}")
except:
    print(f"No previous data found for cbtt_cb_fx.csv")

url = "https://www.central-bank.org.tt/data-center-block/?title=foreign-currency-assets-and-liabilities-monthly&id=86"

r = requests.get(url, headers=hdr, timeout=60)
r.raise_for_status()

soup = BeautifulSoup(r.text, "html.parser")

table = soup.select_one("table#table_1")
if table is None:
    raise ValueError("Could not find table_1 in the page HTML")

# headers
cols = [th.get_text(strip=True) for th in table.select("thead th")]

# rows (only real <tr> rows under tbody)
rows = []
for tr in table.select("tbody tr"):
    tds = [td.get_text(strip=True) for td in tr.select("td")]
    if tds:
        rows.append(tds)

df = pd.DataFrame(rows, columns=cols)

# clean types
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")
df = df.sort_values("Date").reset_index(drop=True)
df.columns = df.columns.astype(str).str.lower()
df.set_index("date", inplace=True)
df.index = pd.to_datetime(df.index)
df.sort_index(inplace=True)

for c in df.columns:
    df[c] = pd.to_numeric(df[c].replace({"n.d.": np.nan, "n.a.": np.nan}), errors="coerce")

def clean_col(c: str) -> str:
    if c == "date":
        return c

    # remove parenthetical content and dash-separated unit suffixes
    c = re.sub(r"\s*\([^)]*\)", "", c)
    c = re.sub(r"\s*-\s*(tt|us)\$\s*millions?", "", c)
    c = c.strip()

    # explicit full-phrase replacements (longest/most-specific first)
    replacements = [
        # cbk sub-components of FCA/FCL
        ("commercial banks: cash items in the process of collection", "cbk_fca_cash_collection"),
        ("commercial banks: due from banks",                          "cbk_fca_due_from_banks"),
        ("commercial banks: due to financial institutions",           "cbk_fcl_due_to_fi"),
        ("commercial banks: investments",                             "cbk_fca_invest"),
        ("commercial banks: other assets",                            "cbk_fca_other"),
        ("commercial banks: other liabilities",                       "cbk_fcl_other"),
        ("commercial banks:cash",                                     "cbk_fca_cash"),
        # cbk aggregate FX positions
        ("commercial banks: net foreign assets",          "cbk_nfa"),
        ("commercial banks: foreign currency assets",     "cbk_fca"),
        ("commercial banks: foreign currency liabilities","cbk_fcl"),
        ("commercial banks: foreign currency deposits",   "cbk_fcd"),
        ("commercial banks: foreign assets",              "cbk_fa"),
        ("commercial banks: foreign liabilities",         "cbk_fl"),
        # cbk FCL deposit types (dash-prefix variant)
        ("commercial banks - demand deposits",   "cbk_fcl_dep_demand"),
        ("commercial banks - savings deposits",  "cbk_fcl_dep_sav"),
        ("commercial banks - time deposits",     "cbk_fcl_dep_time"),
        # FCD totals
        ("total foreign currency deposits - resident and non-resident", "fcd_total_res_nonres"),
        ("total foreign currency deposits - non-resident",              "fcd_total_nonres"),
        ("total foreign currency deposits - resident",                  "fcd_total_res"),
        # FCD by sector/residency (resident-and-non-resident before resident/non-resident)
        ("foreign currency deposits financial institutions - resident and non-resident", "fcd_fi_res_nonres"),
        ("foreign currency deposits financial institutions - non-resident",              "fcd_fi_nonres"),
        ("foreign currency deposits financial institutions - resident",                  "fcd_fi_res"),
        ("foreign currency deposits households - resident and non-resident", "fcd_hh_res_nonres"),
        ("foreign currency deposits households - non-resident",              "fcd_hh_nonres"),
        ("foreign currency deposits households - resident",                  "fcd_hh_res"),
        ("foreign currency deposits incorporated and unincorporated businesses - resident and non-resident", "fcd_biz_res_nonres"),
        ("foreign currency deposits incorporated and unincorporated businesses - non-resident",              "fcd_biz_nonres"),
        ("foreign currency deposits incorporated and unincorporated businesses - resident",                  "fcd_biz_res"),
        ("foreign currency deposits public sector - resident and non-resident", "fcd_pub_res_nonres"),
        ("foreign currency deposits public sector - non-resident",              "fcd_pub_nonres"),
        ("foreign currency deposits public sector - resident",                  "fcd_pub_res"),
    ]
    for old, new in replacements:
        if old in c:
            c = c.replace(old, new)
            break  # each column matches exactly one rule

    # final cleanup
    c = re.sub(r"[^a-z0-9\s_-]", "", c)
    c = re.sub(r"[\s\-]+", "_", c)
    c = re.sub(r"_+", "_", c)
    c = c.strip("_")

    return c

df = df.rename(columns={c: clean_col(c) for c in df.columns})

df.dropna(axis=0, inplace=True, how="all")

print(f"Updated data range: {df.index.min()} to {df.index.max()}")
df.to_csv(f"{PATH_RAW}/cbtt_cb_fx.csv")

## Interest Rates

In [ ]:
try:
    df = pd.read_csv(f"{PATH_RAW}/cbtt_cb_rates.csv", parse_dates=True, index_col="date")
    print(f"Previous data range: {df.index.min()} to {df.index.max()}")
except:
    print(f"No previous data found for cbtt_cb_rates.csv")

url = "https://www.central-bank.org.tt/data-center-block/?title=interest-rates-monthly&id=88"

r = requests.get(url, headers=hdr, timeout=60)
r.raise_for_status()

soup = BeautifulSoup(r.text, "html.parser")

table = soup.select_one("table#table_1")
if table is None:
    raise ValueError("Could not find table_1 in the page HTML")

# headers
cols = [th.get_text(strip=True) for th in table.select("thead th")]

# rows (only real <tr> rows under tbody)
rows = []
for tr in table.select("tbody tr"):
    tds = [td.get_text(strip=True) for td in tr.select("td")]
    if tds:
        rows.append(tds)

df = pd.DataFrame(rows, columns=cols)

# clean types
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")
df = df.sort_values("Date").reset_index(drop=True)
df.columns = df.columns.astype(str).str.lower()
df.set_index("date", inplace=True)
df.index = pd.to_datetime(df.index)
df.sort_index(inplace=True)

for c in df.columns:
    df[c] = pd.to_numeric(df[c].replace({"n.d.": np.nan, "n.a.": np.nan}), errors="coerce")

def clean_col(c: str) -> str:
    if c == "date":
        return c

    # remove parenthetical content
    c = re.sub(r"\s*\([^)]*\)", "", c)
    c = c.strip()

    # cbk aggregate/policy rates (strip prefix first)
    c = re.sub(r"commercial banks:\s*", "cbk_", c)
    c = c.replace("cbk_average basic prime lending rate",                "cbk_rate_prime_avg")
    c = c.replace("cbk_median basic prime lending rate",                 "cbk_rate_prime_med")
    c = c.replace("cbk_demand loans rate",                               "cbk_rate_demand")
    c = c.replace("cbk_ordinary savings deposit rate",                   "cbk_rate_sav_ord")
    c = c.replace("cbk_over 3 - 6 month time",                          "cbk_rate_time_3m_6m")
    c = c.replace("cbk_over 6 - 12 month time",                         "cbk_rate_time_6m_12m")
    c = c.replace("cbk_up to 3 - month time",                           "cbk_rate_time_3m")
    c = c.replace("cbk_overdraft loans rate",                            "cbk_rate_od")
    c = c.replace("cbk_real estate mortgage loan rates",                 "cbk_rate_re_mort")
    c = c.replace("cbk_term loan rates",                                 "cbk_rate_term")
    c = c.replace("cbk_central bank discount rate",                      "cbk_rate_cb_discount")
    c = c.replace("cbk_central bank repo rate",                          "cbk_rate_cb_repo")
    c = c.replace("cbk_weighted average rate on overnight interbank loans", "cbk_rate_interbank_ovnt")
    c = c.replace("cbk_weighted average rate on term interbank loans",      "cbk_rate_interbank_term")

    # individual bank prime rates
    c = c.replace("monthly prime lending rate - ansa bank limited",                                  "rate_prime_ansa")
    c = c.replace("monthly prime lending rate - citibank",                                           "rate_prime_citi")
    c = c.replace("monthly prime lending rate - first caribbean international bank",                 "rate_prime_fcib")
    c = c.replace("monthly prime lending rate - first citizens bank limited",                        "rate_prime_fcb")
    c = c.replace("monthly prime lending rate - jmmb bank",                                         "rate_prime_jmmb")
    c = c.replace("monthly prime lending rate - rbc royal bank",                                     "rate_prime_rbc")
    c = c.replace("monthly prime lending rate - republic bank limited",                              "rate_prime_republic")
    c = c.replace("monthly prime lending rate - scotiabank trinidad and tobago limited",             "rate_prime_scotia")

    # final cleanup
    c = re.sub(r"[^a-z0-9\s_-]", "", c)
    c = re.sub(r"[\s\-]+", "_", c)
    c = re.sub(r"_+", "_", c)
    c = c.strip("_")

    return c

df = df.rename(columns={c: clean_col(c) for c in df.columns})

df.dropna(axis=0, inplace=True, how="all")

print(f"Updated data range: {df.index.min()} to {df.index.max()}")
df.to_csv(f"{PATH_RAW}/cbtt_cb_rates.csv")

## Monetary Aggregates

In [ ]:
try:
    df = pd.read_csv(f"{PATH_RAW}/cbtt_monetary.csv", parse_dates=True, index_col="date")
    print(f"Previous data range: {df.index.min()} to {df.index.max()}")
except:
    print(f"No previous data found for cbtt_monetary.csv")

url = "https://www.central-bank.org.tt/data-center-block/?title=banking-system-monetary-aggregates-monthly&id=223"

r = requests.get(url, headers=hdr, timeout=60)
r.raise_for_status()

soup = BeautifulSoup(r.text, "html.parser")

table = soup.select_one("table#table_1")
if table is None:
    raise ValueError("Could not find table_1 in the page HTML")

# headers
cols = [th.get_text(strip=True) for th in table.select("thead th")]

# rows (only real <tr> rows under tbody)
rows = []
for tr in table.select("tbody tr"):
    tds = [td.get_text(strip=True) for td in tr.select("td")]
    if tds:
        rows.append(tds)

df = pd.DataFrame(rows, columns=cols)

# clean types
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")
df = df.sort_values("Date").reset_index(drop=True)
df.columns = df.columns.astype(str).str.lower()
df.set_index("date", inplace=True)
df.index = pd.to_datetime(df.index)
df.sort_index(inplace=True)

for c in df.columns:
    df[c] = pd.to_numeric(df[c].replace({"n.d.": np.nan, "n.a.": np.nan}), errors="coerce")

def clean_col(c: str) -> str:
    if c == "date":
        return c

    # remove parenthetical content (units and adj flag handled below)
    is_adj = "(adj)" in c
    c = re.sub(r"\s*\([^)]*\)", "", c)
    c = c.strip()

    # explicit full-phrase replacements
    # monetary aggregates (m-2* before m-2 to avoid substring collision)
    c = c.replace("base money m-0",      "m0")
    c = c.replace("narrow money m-1a",   "m1a")
    c = c.replace("money supply m-1c",   "m1c")
    c = c.replace("money supply m-2*",   "m2star")
    c = c.replace("money supply m-2",    "m2")
    c = c.replace("money supply m-3*",   "m3star")
    c = c.replace("money supply m-3",    "m3")
    c = c.replace("quasi money",         "m_quasi")

    # cbk / system components
    c = c.replace("currency in active circulation",              "m_currency_circ")
    c = c.replace("commercial bank reserves with central bank",  "cbk_reserves_cb")
    # cbk foreign currency deposits (specific before generic)
    c = c.replace("commercial banks - foreign currency time deposits",    "cbk_fcd_time")
    c = c.replace("commercial banks - foreign currency demand deposits",  "cbk_fcd_demand")
    c = c.replace("commercial banks - foreign currency savings deposits", "cbk_fcd_sav")
    c = c.replace("commercial banks - foreign currency deposits",         "cbk_fcd")
    c = c.replace("nfis - foreign currency deposits",                     "nfi_fcd")
    # aggregate deposits (these are system-level, no cbk_ prefix)
    c = c.replace("demand deposits", "m_dep_demand")
    c = c.replace("saving deposits", "m_dep_sav")
    c = c.replace("time deposits",   "m_dep_time")

    # append _adj suffix for seasonally/statistically adjusted series
    if is_adj:
        c = c + "_adj"

    # final cleanup
    c = re.sub(r"[^a-z0-9\s_-]", "", c)
    c = re.sub(r"[\s\-]+", "_", c)
    c = re.sub(r"_+", "_", c)
    c = c.strip("_")

    return c

df = df.rename(columns={c: clean_col(c) for c in df.columns})

df.dropna(axis=0, inplace=True, how="all")

print(f"Updated data range: {df.index.min()} to {df.index.max()}")
df.to_csv(f"{PATH_RAW}/cbtt_monetary.csv")

## Loans

In [ ]:
try:
    df = pd.read_csv(f"{PATH_RAW}/cbtt_cb_loans.csv", parse_dates=True, index_col="date")
    print(f"Previous data range: {df.index.min()} to {df.index.max()}")
except:
    print(f"No previous data found for cbtt_cb_loans.csv")

url = "https://www.central-bank.org.tt/data-center-block/?title=commercial-banks-loans-monthly&id=78"

r = requests.get(url, headers=hdr, timeout=60)
r.raise_for_status()

soup = BeautifulSoup(r.text, "html.parser")

table = soup.select_one("table#table_1")
if table is None:
    raise ValueError("Could not find table_1 in the page HTML")

# headers
cols = [th.get_text(strip=True) for th in table.select("thead th")]

# rows (only real <tr> rows under tbody)
rows = []
for tr in table.select("tbody tr"):
    tds = [td.get_text(strip=True) for td in tr.select("td")]
    if tds:
        rows.append(tds)

df = pd.DataFrame(rows, columns=cols)

# clean types
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")
df = df.sort_values("Date").reset_index(drop=True)
df.columns = df.columns.astype(str).str.lower()
df.set_index("date", inplace=True)
df.index = pd.to_datetime(df.index)
df.sort_index(inplace=True)

for c in df.columns:
    df[c] = pd.to_numeric(df[c].replace({"n.d.": np.nan, "n.a.": np.nan}), errors="coerce")

def clean_col(c: str) -> str:
    if c == "date":
        return c

    # detect gross/net qualifiers before stripping parentheses
    is_gross = "(gross)" in c
    is_net   = "(net)" in c

    # remove parenthetical content and dash-separated units
    c = re.sub(r"\s*\([^)]*\)", "", c)
    c = re.sub(r"\s*-\s*(tt|us)\$mn", "", c)

    # strip commercial banks prefix
    c = re.sub(r"commercial banks:\s*", "", c).strip()

    # interest rate columns misplaced in this table
    c = c.replace("weighted average rate on overnight interbank loans", "cbk_rate_interbank_ovnt")
    c = c.replace("weighted average rate on term interbank loans",      "cbk_rate_interbank_term")

    # aggregate totals by loan type (before generic loan-type replacements)
    c = c.replace("total credit card loans",  "loan_cc_total")
    c = c.replace("total leasing loans",      "loan_lease_total")
    c = c.replace("foreign currency loans",   "loan_fx")
    c = c.replace("total loans",              "loan_total")

    # npl and past-due markers (aggregate forms before generic)
    c = c.replace("non performing loans",  "npl_total")
    c = c.replace("total past due loans",  "pastdue_total")
    c = c.replace("non performing",        "npl")
    c = c.replace("total past due",        "pastdue_total")
    c = c.replace("past due",              "pastdue")

    # loan types (bridge/bridging and instalment/installment spelling variants)
    c = c.replace("bridge finance loans",      "loan_bridge")
    c = c.replace("bridging finance loans",    "loan_bridge")
    c = c.replace("credit card loans",         "loan_cc")
    c = c.replace("demand loans",              "loan_demand")
    c = c.replace("discount loans",            "loan_disc")
    c = c.replace("installment loans",         "loan_install")
    c = c.replace("instalment loans",          "loan_install")
    c = c.replace("leasing loans",             "loan_lease")
    c = c.replace("lease financing loans",     "loan_lease")
    c = c.replace("real estate mortgages loans","loan_re_mort")
    c = c.replace("real estate mortgage loans","loan_re_mort")
    c = c.replace("overdraft loans",           "loan_od")
    c = c.replace("time loans",                "loan_time")

    # aging buckets (longest matches first)
    c = c.replace("over 12-24 months", "12m_24m")
    c = c.replace("over 24 months",    "24m_plus")
    c = c.replace("over 1-3 months",   "1m_3m")
    c = c.replace("over 3-6 months",   "3m_6m")
    c = c.replace("over 6-12 months",  "6m_12m")
    c = c.replace("1 month",           "1m")

    # borrower types (unincorporated before incorporated)
    c = c.replace("unincorporated enterprises", "uninc")
    c = c.replace("unincorporated businesses",  "uninc")
    c = c.replace("incorporated enterprises",   "inc")
    c = c.replace("incorporated businesses",    "inc")
    c = c.replace("financial institutions",     "fi")
    c = c.replace("consumers",                  "cons")
    c = c.replace("public sector",             "pub")

    # gross/net suffix
    if is_gross:
        c = c + " gross"
    elif is_net:
        c = c + " net"

    # final cleanup
    c = re.sub(r"[^a-z0-9\s_-]", "", c)
    c = re.sub(r"[\s\-]+", "_", c)
    c = re.sub(r"_+", "_", c)
    c = c.strip("_")

    # add cbk_ prefix where not already present
    if not c.startswith("cbk_"):
        c = "cbk_" + c

    return c

df = df.rename(columns={c: clean_col(c) for c in df.columns})

df.dropna(axis=0, inplace=True, how="all")

print(f"Updated data range: {df.index.min()} to {df.index.max()}")
df.to_csv(f"{PATH_RAW}/cbtt_cb_loans.csv")

## Repo Rate

In [ ]:
try:
    df = pd.read_csv(f"{PATH_RAW}/cbtt_repo.csv", parse_dates=True, index_col="date")
    print(f"Previous data range: {df.index.min()} to {df.index.max()}")
except:
    print(f"No previous data found for cbtt_repo.csv")

url = "https://www.central-bank.org.tt/data-center-block/?title=selected-economic-indicators-quarterly&id=3"

r = requests.get(url, headers=hdr, timeout=60)
r.raise_for_status()

soup = BeautifulSoup(r.text, "html.parser")

table = soup.select_one("table#table_1")
if table is None:
    raise ValueError("Could not find table_1 in the page HTML")

# headers
cols = [th.get_text(strip=True) for th in table.select("thead th")]

# rows (only real <tr> rows under tbody)
rows = []
for tr in table.select("tbody tr"):
    tds = [td.get_text(strip=True) for td in tr.select("td")]
    if tds:
        rows.append(tds)

df = pd.DataFrame(rows, columns=cols)

# clean types
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")
df = df.sort_values("Date").reset_index(drop=True)
df.columns = df.columns.astype(str).str.lower()
df.set_index("date", inplace=True)
df.index = pd.to_datetime(df.index)
df.sort_index(inplace=True)

for c in df.columns:
    df[c] = pd.to_numeric(df[c].replace({"n.d.": np.nan, "n.a.": np.nan}), errors="coerce")

# keep only the repo rate column and rename it
repo_col = [c for c in df.columns if "repo rate" in c.lower()]
if not repo_col:
    raise ValueError("Repo Rate column not found in table")
df = df[[repo_col[0]]].rename(columns={repo_col[0]: "repo_rate"})

df.dropna(axis=0, inplace=True, how="all")

print(f"Updated data range: {df.index.min()} to {df.index.max()}")
df.to_csv(f"{PATH_RAW}/cbtt_repo.csv")

## Exchange Rates

In [ ]:
try:
    df = pd.read_csv(f"{PATH_RAW}/cbtt_fx_day.csv", parse_dates=True, index_col="date")
    print(f"Previous data range: {df.index.min()} to {df.index.max()}")
except:
    print(f"No previous data found for cbtt_fx_day.csv")


url = "https://www.central-bank.org.tt/data-center-block/?title=exchange-rates-daily&id=106"


r = requests.get(url, headers=hdr, timeout=60)
r.raise_for_status()

soup = BeautifulSoup(r.text, "html.parser")

table = soup.select_one("table#table_1")
if table is None:
    raise ValueError("Could not find table_1 in the page HTML")

# headers
cols = [th.get_text(strip=True) for th in table.select("thead th")]

# rows (only real <tr> rows under tbody)
rows = []
for tr in table.select("tbody tr"):
    tds = [td.get_text(strip=True) for td in tr.select("td")]
    if tds:
        rows.append(tds)

df = pd.DataFrame(rows, columns=cols)

# clean types
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")
df = df.sort_values("Date").reset_index(drop=True)
df.columns = df.columns.astype(str).str.lower()
df.set_index("date", inplace=True)
df.index = pd.to_datetime(df.index)
df.sort_index(inplace=True)

for c in df.columns:
    df[c] = pd.to_numeric(df[c].replace({"n.d.": np.nan, "n.a.": np.nan}), errors="coerce")


def clean_col(c: str) -> str:
    if c == "date":
        return c
    c = c.replace(" buying rate",  "_buy")
    c = c.replace(" selling rate", "_sell")
    return "fx_" + c

df = df.rename(columns={c: clean_col(c) for c in df.columns})

df.dropna(axis=0, inplace=True, how="all")

print(f"Updated data range: {df.index.min()} to {df.index.max()}")
df.to_csv(f"{PATH_RAW}/cbtt_fx_day.csv")

## National Summary Data Page (NSDP)

### GDP National Accounts

In [ ]:
try:
    df_prev = pd.read_csv(f"{PATH_RAW}/cbtt_gdp_datapack.csv", parse_dates=True, index_col="date")
    print(f"Previous data range: {df_prev.index.min()} to {df_prev.index.max()}")
except:
    print("No previous data found for cbtt_gdp_datapack.csv")

# ---- 1. Find latest XLSM on the datapack page ----
page_url = "https://www.central-bank.org.tt/resources-category/publications-and-research/all-reports/economic-datapack-report/"

r = requests.get(page_url, headers=hdr, timeout=60)
r.raise_for_status()
soup = BeautifulSoup(r.text, "html.parser")

xlsm_links = sorted({
    urljoin(page_url, a["href"])
    for a in soup.find_all("a", href=True)
    if a["href"].lower().endswith(".xlsm") or a["href"].lower().endswith(".xlsx")
})

if not xlsm_links:
    raise ValueError("No XLSM/XLSX found on the datapack page")

latest_url = xlsm_links[-1]
print(f"Latest datapack: {latest_url}")

# ---- 2. Download the file ----
r2 = requests.get(latest_url, headers=hdr, timeout=120)
r2.raise_for_status()
file_bytes = r2.content

# ---- 3. Read raw (no headers) to inspect structure ----

raw = pd.ExcelFile(BytesIO(file_bytes))
SHEET  = [ col for col in raw.sheet_names if "DomesticAct_Quarterly" in col ][0]
raw = pd.read_excel(raw , sheet_name=SHEET, header=None, engine="openpyxl", skiprows=8)

# ---- 4. Build column names from 3-row hierarchical header ----
# rows 0-2 are header levels; row 3+ is data

hdr_block = raw.iloc[:3, :].copy()

# Forward-fill row 0 so the merged "QGDP" label covers all its sub-columns
row0_ffill = hdr_block.iloc[0].astype(str).ffill()

# QGDP columns: wherever row 0 (ffilled) contains the QGDP label
qgdp_mask = row0_ffill.str.contains("Gross Domestic Product|QGDP", case=False, na=False)
qgdp_cols  = [c for c in raw.columns if qgdp_mask[c]]

if not qgdp_cols:
    raise ValueError("Could not find QGDP columns in header row 0")

def build_name(col_idx):
    # Combine row-1 + row-2 labels; skip group header and 'of which:'
    parts = []
    for row_i in [1, 2]:
        val = str(hdr_block.iloc[row_i, col_idx]).strip()
        if val.lower() in ("nan", "", "of which:"):
            continue
        parts.append(val)
    name = " ".join(parts)
    # abbreviate verbose phrases before generic cleanup
    name = re.sub(r"excl\.?\s*[^)]*",                      "",         name, flags=re.I)
    name = re.sub(r"\(.*?\)",                               "",         name)
    name = name.replace("Non-Energy",                         "nonenergy")
    name = name.replace("Trade and Repairs",                  "trade")
    name = name.replace("Financial and Insurance Activities", "fin_ins")
    name = name.replace("Manufacturing",                      "manuf")
    # generic cleanup
    name = name.lower()
    name = re.sub(r"[^a-z0-9\s]", " ", name)
    name = re.sub(r"\s+",          "_", name.strip())
    name = re.sub(r"_+",            "_", name).strip("_")
    return "qgdp_" + name if name else f"qgdp_col{col_idx}"

col_map = {c: build_name(c) for c in qgdp_cols}
col_map[0] = "date"

print("Column names:", [col_map[c] for c in qgdp_cols])

# ---- 5. Extract data (rows 3+), keep date + QGDP only ----
df = raw.iloc[3:, [0] + qgdp_cols].copy()
df.columns = ["date"] + [col_map[c] for c in qgdp_cols]

df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df[df["date"].notna()].copy()

for c in df.columns[1:]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df = df.sort_values("date").set_index("date")
df.sort_index(inplace=True)
df.dropna(how="all", inplace=True)

print(f"Updated data range: {df.index.min()} to {df.index.max()}")
df.to_csv(f"{PATH_RAW}/cbtt_gdp_datapack.csv")

In [ ]:
# National Accounts
url = "https://cso.gov.tt/wp-content/uploads/SDMX/NAG_TTO.xml"

# ---- 1) Download and parse XML ----
r = requests.get(url)
r.raise_for_status()

root = ET.fromstring(r.content)

# ---- 2) Collect long format data ----
rows = []
for series in root.findall(".//{*}Series"):
    indicator = series.attrib.get("INDICATOR")
    
    for obs in series.findall("{*}Obs"):
        rows.append({
            "date": obs.attrib.get("TIME_PERIOD"),
            "indicator": indicator,
            "value": pd.to_numeric(obs.attrib.get("OBS_VALUE"), errors="coerce")
        })

df = pd.DataFrame(rows)
df = df[df["date"].str.match(r"^\d{4}-Q[1-4]$")]

q = pd.PeriodIndex(df["date"], freq="Q")
df["date"] = q.asfreq("M", "end").to_timestamp()
df["date"] = pd.to_datetime(df["date"])

df = df.pivot(index="date", columns="indicator", values="value")

print(f"QGDP Updated data: {df.index.max()}") 
df.to_csv(f"{PATH_RAW}/cbtt_gdp.csv")

In [ ]:
# Combine historic data with most recent data

url = "https://cso.gov.tt/subjects/national-accounts/"

response = requests.get(url, headers=hdr)

soup = BeautifulSoup(response.text, "html.parser")

xlsx_links = []

for link in soup.find_all("a", href=True):
    href = link["href"]
    if ".xlsx" in href.lower():
        full_url = urljoin(url, href)  # handle relative links
        xlsx_links.append(full_url)

xlsx_links = sorted(set(xlsx_links))
xlsx_links = [ l for l in xlsx_links if "constant-price-qgdp" in l.lower() and "tt-mill" in l.lower() ]

if len(xlsx_links)>0 :
    response = requests.get(xlsx_links[0], headers=hdr)
    xls = pd.ExcelFile(BytesIO(response.content))

    dfi = pd.read_excel(xls, sheet_name=xls.sheet_names[0], skiprows=3)
    # Keep only the GDP at producer prices row from the Quarter/Year column:
    dfi = dfi[dfi["Quarter/Year"].str.contains("GDP at producer prices", case=False, na=False)]

    dfi.columns = dfi.columns.astype(str).str.lower()

    dfi = dfi.melt(id_vars="quarter/year", var_name="date", value_name="NGDP_PA_R_XDC")

    dfi = dfi[['date', 'NGDP_PA_R_XDC']]
    dfi.dropna(inplace=True)
    # 1) Turn "YYYY-#"(quarter number) into "YYYYQ#" (a proper quarterly period label)
    q = pd.PeriodIndex(dfi["date"].astype(str).str.replace("-", "Q", regex=False), freq="Q")

    # 2) Last month of quarter, then timestamp at first day of that month
    dfi["date"] = q.asfreq("M", "end").to_timestamp()

    dfi.set_index("date", inplace=True)

    dfi = dfi[dfi.index > df.index.max()]

    df = dfi.combine_first(df)

print(f"Updated data range: {df.index.min()} to {df.index.max()}") 
df.to_csv(f"{PATH_RAW}/cbtt_gdp.csv")

### Consumer Price Index

In [ ]:
url = "https://cso.gov.tt/wp-content/uploads/SDMX/CPI_TTO.xml"

# ---- 1) Download and parse XML ----
r = requests.get(url)
r.raise_for_status()

root = ET.fromstring(r.content)

# ---- 2) Collect long format data ----
rows = []
for series in root.findall(".//{*}Series"):
    indicator = series.attrib.get("INDICATOR")
    
    for obs in series.findall("{*}Obs"):
        rows.append({
            "date": obs.attrib.get("TIME_PERIOD"),
            "indicator": indicator,
            "value": pd.to_numeric(obs.attrib.get("OBS_VALUE"), errors="coerce")
        })

df = pd.DataFrame(rows)
df['date'] = pd.to_datetime(df.date)

df = df.pivot(index="date", columns="indicator", values="value")

print(f"Updated data range: {df.index.min()} to {df.index.max()}") 
df.to_csv(f"{PATH_RAW}/cbtt_cpi.csv")

### Central Government Operations

In [ ]:
url = "https://www.finance.gov.tt/wp-content/uploads/2021/09/CGO_TTO.xml"

# ---- 1) Download and parse XML ----
r = requests.get(url)
r.raise_for_status()

root = ET.fromstring(r.content)

# ---- 2) Collect long format data ----
rows = []
for series in root.findall(".//{*}Series"):
    indicator = series.attrib.get("INDICATOR")
    
    for obs in series.findall("{*}Obs"):
        rows.append({
            "date": obs.attrib.get("TIME_PERIOD"),
            "indicator": indicator,
            "value": pd.to_numeric(obs.attrib.get("OBS_VALUE"), errors="coerce")
        })

df = pd.DataFrame(rows)
df['date'] = pd.to_datetime(df.date)

df = df.pivot(index="date", columns="indicator", values="value")
print(f"Updated data range: {df.index.min()} to {df.index.max()}") 
df.to_csv(f"{PATH_RAW}/cbtt_cgo.csv")

In [ ]:
df

# International Data

## FRED

In [ ]:
api_key = "72304a91a13cc0ea6c610ae5845c7493"
series = ["POILWTIUSDM", "EXCAUS", "DEXUSEU", "DEXUSUK", "SP500", "DJIA", "NASDAQCOM"]

df = pd.DataFrame()
for serie in series :
    #&realtime_end=9999-12-31
    url = f"https://api.stlouisfed.org/fred/series/observations?series_id={serie}&api_key={api_key}&file_type=json"
    response = requests.get(url)
    response = response.json()
    dfi = pd.DataFrame(response['observations'])
    dfi = dfi[['value', 'date']]
    dfi.set_index('date', inplace=True)
    dfi.index = pd.to_datetime(dfi.index)
    dfi.rename(columns = { "value" : f"fred_{serie}" }, inplace=True)
    dfi[f"fred_{serie}"] = dfi[f"fred_{serie}"].apply(pd.to_numeric, errors='coerce')
    df = df.merge(dfi, left_index=True, right_index=True, how='outer')
    df = pd.DataFrame(df.resample("D").mean())
    #df = pd.concat([df, dfi], axis=1)
    df.sort_index(inplace=True)
    
df.to_csv(f"{PATH_RAW}/fred_data.csv")
print(f"FRED Updated: {df.index.max()}")

## EIA

In [ ]:
from io import BytesIO
'''
Access the EIA and download the xls file that has all data.
'''
url = "https://www.eia.gov/dnav/pet/hist/LeafHandler.ashx?n=PET&s=RBRTE&f=D"

r = requests.get(url=url, verify=None).content
soup = BeautifulSoup(r)
links = [ link.get("href") for link in soup('a') if link.get("href") is not None ]
link = [ link for link in links if "xls" in link ][0].replace("../","")
get = f"https://www.eia.gov/dnav/pet/{link}"
df = pd.ExcelFile(get)

df = pd.read_excel( df , sheet_name=[ sheet for sheet in df.sheet_names if "data" in sheet.lower() ][0], skiprows=2)
df.columns = df.columns.str.lower()
df.set_index('date', inplace=True)
df = df.rename(columns=lambda x: "brent price" if "brent" in x else x)
df.index = pd.to_datetime(df.index)

df.to_csv(f"{PATH_RAW}/eia_brent.csv")
print(f"EIA  Updated: {df.index.max()}")

## USA Census Trade

In [ ]:

'''
Access the US Census and dowload the xls file with all the data
'''
url = "https://www.census.gov/foreign-trade/balance/c2360.html"

r = requests.get(url=url, verify=None).content
soup = BeautifulSoup(r)
links = [ link.get("href") for link in soup('a') if link.get("href") is not None ]
get = [ link for link in links if "xls" in link ]
get = f"https://www.census.gov/{get[0]}"

#df = pd.read_excel(get)
df = pd.read_excel(BytesIO(requests.get(get, headers=hdr).content))
df = df[df['CTYNAME'].str.contains("Trinidad and Tobago")]

df = df.drop(['CTY_CODE', 'CTYNAME', 'IYR', 'EYR'], axis=1)
df.columns = df.columns.str.lower()
df = pd.melt(df, id_vars="year", var_name="month", value_name='trade')

df['variable'] = df['month'].str[0]
df['month'] = df['month'].str[1:]
df['variable'] = df['variable'].replace({'i': 'USA_imp_TTO', 'e': 'USA_exp_TTO'})
df.columns = df.columns.astype(str).str.strip()

df['date'] = pd.to_datetime(df['year'].astype(str)+df['month'], format="%Y%b") + pd.offsets.MonthEnd(0)
df = df.set_index('date')[['trade', 'variable']]

df = df.pivot(columns='variable', values='trade')
df = df[df["USA_exp_TTO"] != 0]

df.to_csv(f"{PATH_RAW}/uscensus_trade.csv")
print(f"US Census Updated: {df.index.max()}")

## Canada Trade

In [ ]:
import zipfile
import io
import shutil
def extract_year(url):
    match = re.search(r'(20\d{2})', url)
    if match:
        return int(match.group(1))
    return None

try :
    df = pd.read_csv(f"{PATH_RAW}/canada_imports.csv", index_col='date', parse_dates=True)
    #2022-2025 Files
    url = "https://open.canada.ca/data/api/action/package_show?id=2909a648-5753-4924-878a-b069392d9cde"
except :
    df = pd.DataFrame()
    url = [
        "https://open.canada.ca/data/api/action/package_show?id=ed551c1b-ba5d-4391-a08e-8c154fb2e629", 
        "https://open.canada.ca/data/api/action/package_show?id=7b9ef472-e806-42f8-8e2a-c8d954e250e0", 
        "https://open.canada.ca/data/api/action/package_show?id=d0c43a70-3390-4879-8f1d-b602bb0eba3d", 
        "https://open.canada.ca/data/api/action/package_show?id=6f3fd3d9-5a35-47a8-a45a-4b2a8c957453", 
        "https://open.canada.ca/data/api/action/package_show?id=b1126a07-fd85-4d56-8395-143aba1747a4" ,
        "https://open.canada.ca/data/api/action/package_show?id=2909a648-5753-4924-878a-b069392d9cde"
        ]
    

if isinstance(url, list) :
    links = []
    for u in url :
        r = requests.get(url=u, verify=None)
        resources = r.json()["result"]["resources"]
        l = [ res["url"] for res in resources if "Imp" in res["url"] ]
        links = links + l
else :
    r = requests.get(url=url, verify=None)
    resources = r.json()["result"]["resources"]
    links = [ res["url"] for res in resources if "Imp" in res["url"] ]
    links = [ link for link in links if extract_year(link.split("/")[-1]) >= df.index.max().year ]
print(links)

In [ ]:
for link in links : 
    # Download the Zip file
    response = requests.get(link)
    with zipfile.ZipFile(io.BytesIO(response.content)) as z:
        z.extractall(f"{PATH_RAW}")  # Extract to a folder
        print("ZIP extracted successfully!")

    # Extract file 022 (imports by product)
    file_name = [ x for x in z.namelist() if "N022" in x ][0]
    #file_name = [ x for x in z.namelist() if "N022" in x.split("/")[-1] ][0]
    file_name = f"{PATH_RAW}/{file_name}"
    # Open the file
    dfi = pd.read_csv(file_name)

    # Clean variable names
    for col in dfi.columns :
        if "YearMonth" in col :
            dfi = dfi.rename(columns = {col : 'date'} )
        elif "Country" in col :
            dfi = dfi.rename(columns = {col : 'partner'} )
        elif "Value" in col :
            dfi = dfi.rename(columns = {col : 'value'} )
        elif "Quantity" in col :
            dfi = dfi.rename(columns = {col : 'quantity'} )
        elif "Unit" in col :
            dfi = dfi.rename(columns = {col : 'unit'} )

    # Find Bahamas
    dfi = dfi[dfi['partner']=='TT']
    # Clean date
    dfi['date'] = pd.to_datetime(pd.to_datetime(dfi['date'] , format="%Y%m").dt.strftime('%Y-%m-%d'))
    dfi = dfi.set_index('date')

    # Sum all monthly values
    dfi = dfi.resample('MS')[['value']].sum()
    dfi.rename(columns = {"value" : 'imports_can'} , inplace=True)
    df = df[~df.index.isin(dfi.index.unique())]
    df = pd.concat([df, dfi], axis=0)

    try :
        #folder_path = os.path.dirname(os.path.abspath(file_name))
        os.remove(file_name)
    except :
        continue

df = df.sort_index()
print(f"Updated: {df.index.max()}")
df.to_csv(f'{PATH_RAW}/canada_imports.csv')

In [ ]:
try :
    df = pd.read_csv(f"{PATH_RAW}/canada_exports.csv", index_col='date', parse_dates=True)
    #2022-2025 Files
    url = "https://open.canada.ca/data/api/action/package_show?id=2909a648-5753-4924-878a-b069392d9cde"
except :
    df = pd.DataFrame()
    url = [
        "https://open.canada.ca/data/api/action/package_show?id=ed551c1b-ba5d-4391-a08e-8c154fb2e629", 
        "https://open.canada.ca/data/api/action/package_show?id=7b9ef472-e806-42f8-8e2a-c8d954e250e0", 
        "https://open.canada.ca/data/api/action/package_show?id=d0c43a70-3390-4879-8f1d-b602bb0eba3d", 
        "https://open.canada.ca/data/api/action/package_show?id=6f3fd3d9-5a35-47a8-a45a-4b2a8c957453", 
        "https://open.canada.ca/data/api/action/package_show?id=b1126a07-fd85-4d56-8395-143aba1747a4" ,
        "https://open.canada.ca/data/api/action/package_show?id=2909a648-5753-4924-878a-b069392d9cde"
        ]

if isinstance(url, list) :
    links = []
    for u in url :
        r = requests.get(url=u, verify=None)
        resources = r.json()["result"]["resources"]
        l = [ res["url"] for res in resources if "tot_exp" in res["url"].lower() ]
        links = links + l
else :
    r = requests.get(url=url, verify=None)
    resources = r.json()["result"]["resources"]
    links = [ res["url"] for res in resources if "tot_exp" in res["url"].lower() ]
    links = [ link for link in links if extract_year(link.split("/")[-1]) >= df.index.max().year ]

In [ ]:
for link in links : 
    response = requests.get(link)
    
    with zipfile.ZipFile(io.BytesIO(response.content)) as z:
        z.extractall(f"{PATH_RAW}")  # Extract to a folder
        print("ZIP extracted successfully!")
        
    file_name = [ x for x in z.namelist() if "N021" in x ][0]
    #file_name = [ x for x in z.namelist() if "N021" in x.split("/")[-1] ][0]
    file_name = f"{PATH_RAW}/{file_name}"
    dfi = pd.read_csv(file_name)
    for col in dfi.columns :
        if "YearMonth" in col :
            dfi = dfi.rename(columns = {col : 'date'} )
        elif "Country" in col :
            dfi = dfi.rename(columns = {col : 'partner'} )
        elif "Value" in col :
            dfi = dfi.rename(columns = {col : 'value'} )
        elif "Quantity" in col :
            dfi = dfi.rename(columns = {col : 'quantity'} )
        elif "Unit" in col :
            dfi = dfi.rename(columns = {col : 'unit'} )
    dfi = dfi[dfi['partner']=='TT']
    dfi['date'] = pd.to_datetime(pd.to_datetime(dfi['date'] , format="%Y%m").dt.strftime('%Y-%m-%d'))
    dfi = dfi.set_index('date')
    dfi = dfi.resample('MS')[['value']].sum()
       
    dfi.rename(columns = {"value" : 'exports_can'} , inplace=True)
    df = df[~df.index.isin(dfi.index.unique())]
    df = pd.concat([df, dfi], axis=0)

    try :
        #folder_path = os.path.dirname(os.path.abspath(file_name))
        os.remove(file_name)
        #if os.path.exists(folder_path): shutil.rmtree(folder_path)
    except :
        continue

df = df.sort_index()
print(f"Updated: {df.index.max()}")
df.to_csv(f'{PATH_RAW}/canada_exports.csv')

In [ ]:
# Clean up extracted Statistics Canada files
for f in glob.glob(os.path.join(PATH_RAW, "ODPF*")):
    os.remove(f)
for f in glob.glob(os.path.join(PATH_RAW, "Note*.docx")):
    os.remove(f)
for d in glob.glob(os.path.join(PATH_RAW, "CIMT-CICM_*")):
    shutil.rmtree(d)
print("Cleanup done.")

## European Union

In [ ]:
'''
Download data from eurostat.
eurostat package is required.
'''
# conda install conda-forge::eurostat
import eurostat
dataset = 'ds-045409'
params = {
    'format': 'TSV',        # Use uncompressed TSV format for simplicity
    'geo': 'EU',            # Specify the region (EU)
    'partner': 'TT',        # Specify the partner (Trinidad and Tobago)
    'time': '2000',         # You can specify a smaller time range (e.g., 2021)
    'trade': '1',           # You can filter for imports (1) or exports (2)
    'product': '0',         # You can specify product codes if needed
    'compressed': 'false',  # Disable compression
}
df = eurostat.get_data_df(dataset,
                       filter_pars = {
                           'reporter' : 'EU' ,
                           'partner' : 'TT' ,
                           'indicators' : 'VALUE_IN_EUROS',
                           'freq' : 'M' ,
                           'product' : 'TOTAL'
                       })
df.drop(['freq', 'reporter', 'partner', 'product'], axis=1, inplace=True)

df = pd.melt(df,
        id_vars=['flow', 'indicators\\TIME_PERIOD'],
        var_name='date', value_name='value')

df['date'] = pd.to_datetime(pd.to_datetime(df['date'] , format="%Y-%m").dt.strftime('%Y-%m-%d'))
df['flow'] = df['flow'].map({"1": 'imp_eu', "2": 'exp_eu'})
df.set_index('date', inplace=True)
df = df.pivot_table(index=df.index, columns='flow', values='value')

print(f"Updated: {df.index.max()}")
df.to_csv(f'{PATH_RAW}/eu_trade.csv')

## UK Trade

In [ ]:
# Prepare all functions to read data.

import pprint as pp
import logging

logger = logging.getLogger(__name__)

ROOT_URL = "https://api.beta.ons.gov.uk/v1/"


def get_list_of_datasets():
    """Get list of all datasets available from API.
    Currently (August 2021), there are 41 available datasets.

    Returns
    -------
    list of dicts
        Metadata objects for each available dataset.
    """
    num_to_get = 100
    datasets = []

    offset = 0
    while len(datasets) < num_to_get:
        r = requests.get(ROOT_URL + "datasets", params={"offset": offset})
        results = r.json()
        [logger.info(item.get("title")) for item in results.get("items")]
        datasets.extend(results.get("items"))
        num_retrieved = results.get("count")
        offset += num_retrieved
        if num_retrieved == 0:
            break
    logger.info(f"\nFound {len(datasets)} datasets")
    return datasets


def get_dataset_by_name(datasets, target_name):
    """Get a dataset by matching on its title (or part of the title).
    Returns the first dataset object whose name contains the given target_name string.

    Parameters
    ----------
    datasets : List
        List of dataset objects
    target_name : str
        name (or partial name) of target dataset

    Returns
    -------
    dict
        Dataset object (or None, if no match is found)
    """
    for ds in datasets:
        if target_name.lower() in ds.get("title").lower():
            logger.info(f"Found dataset '{ds.get('title')}'")
            return ds
    logger.info(f"No dataset found containing '{target_name}'")
    return None


def get_edition(dataset, prefered_edition="time-series"):
    """Get one edition of a dataset. If no preferred edition is
    specified, return the most recent one.

    Parameters
    ----------
    dataset : dict
        dataset metadata
    prefered_edition : str, optional
        name of edition, by default "time-series"

    Returns
    -------
    str
        URL of edition
    """
    editions_url = dataset.get("links").get("editions").get("href")
    r = requests.get(editions_url)
    results = r.json()
    for row in results.get("items"):
        if row.get("edition") == prefered_edition:
            edition = row.get("links").get("latest_version").get("href")
            return edition

    # Default to latest version, if requested version is not found.
    latest_version = dataset.get("links").get("latest_version").get("href")
    return latest_version


def get_dimensions(edition_url):
    """Builds dictionary of all valid options for all dimensions of a given dataset,
    with descriptions for each option.
    Individual obvserviations can later be obtained by choosing from these options.
    Ranges can later be obtained by replacing one dimesion with the wildcard '*'.

    Parameters
    ----------
    dataset : dict
        single dataset

    Returns
    -------
    dict of dicts
        map of {dimensions:{acceptable_values:description}}
    """
    valid_dimensions = {}
    r = requests.get(edition_url + "/dimensions")
    results = r.json()
    for dimension in results.get("items"):
        logger.info(f'{dimension.get("name")}: \t{dimension.get("label")}')
        dim_id = dimension.get("links").get("options").get("id")
        options_url = f"{edition_url}/dimensions/{dim_id}/options"

        sr = requests.get(options_url, params={"limit": 50})
        sresults = sr.json()
        # TODO! Could add in paging here, as there *could* be multiple pages of valid options.
        logger.info(f"\tHas {sresults.get('count')} options")
        # valid_options = [item.get("option") for item in sresults.get("items")]
        option_descriptions = {
            item.get("option"): item.get("label") for item in sresults.get("items")
        }
        logger.info(f'{dimension.get("name")}: {option_descriptions}')
        valid_dimensions[dimension.get("name")] = option_descriptions

    return valid_dimensions


def choose_dimensions(valid_dims, overrides={}):
    """For each dimension, choose a single valid option (except for 'time', where
    we use the wildcard '*' to get the whole time-series.)
    If not specified, choose the first valid option for each dimension.

    Parameters
    ----------
    valid_dims : dict
        map of lists of valid dimension values
    overrides : dict, optional
        selected dimensions

    Returns
    -------
    dict
        final choice of dimensions
    """
    # By default, choose first valid item for all dimensions; then override where needed:
    chosen_dimensions = {k: next(iter(v.keys())) for k, v in valid_dims.items()}
    # get whole time-series, not just a single point in time:
    chosen_dimensions["time"] = "*"
    chosen_dimensions.update(overrides)
    return chosen_dimensions


def get_observations(edition_url, dimensions):
    """[summary]

    Parameters
    ----------
    edition_url : str
        URL of this edition of the data
    dimensions : dict
        dimensions specifying slice of data required

    Returns
    -------
    pd.Dataframe
        Summary of data, with columns "id" (time) and "observation" (value)
    """
    r = requests.get(edition_url + "/observations", params=dimensions)
    results = r.json()
    summary = []
    for observation in results.get("observations"):
        id = observation.get("dimensions").get("Time").get("id")
        summary.append({"id": id, "observation": observation.get("observation")})
    df = pd.DataFrame(summary)
    return df


def get_timeseries(dataset_name, dimension_values):
    """Get a specified dataset time-series, with a given set of dimensions.
    NB: dataframe is not sorted.

    Parameters
    ----------
    dataset_name : str
        Descriptive name of data set.
    dimension_values : dict
        set of valid dimensions for this dataset.
        If set to "None", then return set of valid dimesions.

    Returns
    -------
    Either:
        dict
            Set of valid dimensions, if None specified in function call
    or:
        dataframe
            containing time series
        dict
            dataset metadata
        str
            url of this edition of the data
    """
    dss = get_list_of_datasets()
    ds = get_dataset_by_name(dss, dataset_name)
    edition_url = get_edition(ds)
    valid_dims = get_dimensions(edition_url)
    logger.info(valid_dims)
    if dimension_values is None:
        return valid_dims
    chosen_dimensions = choose_dimensions(valid_dims, dimension_values)
    df = get_observations(edition_url, chosen_dimensions)
    logger.info(df.shape)
    return df, ds, edition_url


def demo():
    print("=" * 70)
    print("List of available datasets:")
    dss = get_list_of_datasets()
    [print(item.get("title")) for item in dss]

    print("=" * 70)
    dataset_name = "UK Labour Market"
    print(f"Valid options for dimensions for the {dataset_name}, with descriptions")
    # Get the set of valid dimensions for the Labour Market set.
    # E.g. list of valid age groups, economic activity categories etc.

    dimensions = get_timeseries(dataset_name, None)
    pp.pprint(dimensions)
    print("\n")

    # We've now selected specific dimensions for our request:
    labour_market_dimensions = {
        "economicactivity": "in-employment",
        "agegroups": "16+",
        "seasonaladjustment": "seasonal-adjustment",
        "sex": "all-adults",
        "unitofmeasure": "rates",
    }
    print(f"Chosen dimensions for the {dataset_name}")
    pp.pprint(labour_market_dimensions, indent=4)

    df_labour = get_timeseries(dataset_name, labour_market_dimensions)[0]
    df_labour["year"] = (
        df_labour["id"].str[-4:].astype(int)
    )  # extract year as last 4 digits of row id
    df_labour = df_labour.sort_values("year")
    print("")
    print(df_labour)
    print("\n")

    # Repeat the process for a second time series, GDP, with specific dimensions:
    print("=" * 70)
    gdp_dataset_name = "annual GDP"
    gdp_dimensions = {
        "geography": "UK0",
        "unofficialstandardindustrialclassification": "A--T",
    }
    print(f"Chosen dimensions for the {dataset_name}")
    df_gdp = get_timeseries(gdp_dataset_name, gdp_dimensions)[0]
    df_gdp = df_gdp.sort_values("id")
    pp.pprint(gdp_dimensions, indent=4)
    print("")
    print(gdp_dataset_name)
    print(df_gdp)

    print("=" * 70)
    print("End of demo!")

In [ ]:
#dss = get_list_of_datasets()
dataset_name = "Trade in goods: country by commodity"
#dss = [ item for item in dss if item.get("title")==dataset_name ]
#dimensions = get_timeseries(dataset_name, None)
#dimensions

df = pd.DataFrame()

for var in ['exp_uk' , 'imp_uk'] :
    if var == "exp_uk" : t = 'EX'
    elif var == "imp_uk" : t = 'IM'
    params = {
        "countriesandterritories": "TT",
        "standardindustrialtradeclassification": "T",
        #"standardindustrialtradeclassification": "0",
        'geography': 'K02000001' ,
        'direction': t ,
        }
    dfi = get_timeseries(dataset_name, params)
    dfi = dfi[0]
    dfi['date'] = pd.to_datetime(dfi['id'], format="%b-%y")
    dfi = dfi.set_index('date')
    dfi.rename(columns = { 'observation' : var} , inplace=True)
    dfi = dfi[[var]]
    dfi[var] = pd.to_numeric(dfi[var])
    
    df = pd.concat( [df, dfi], axis=1)

print(f"Updated: {df.index.max()}")
df.to_csv(f'{PATH_RAW}/uk_trade.csv')